# 舊版程式

## SubclassSegformer

In [ ]:
import os
import torch
import numpy as np
from tqdm import tqdm
from dataset import RippleFeatureDataset
# 確保引入原始版本模型類別，以便 torch.load 進行反序列化
from semseg.models.subclass_segformer import SubclassSegFormer
from util.analysis_algs import *

def run_unseen_inference_no_prompt():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- 1. 配置與路徑 ---
    UNSEEN_DOMAINS = {
        'F': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG6235/label',
        'G': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG5433/label',
        'H': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201022/label',
        'I': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201211/label',
        'J': 'C:/Users/user/PycharmProjects/meta_expand_test/stereo0420/label'
    }

    # 學長權重路徑
    weight_path = '../output_mix5_sb2L2AW_new64_aug10_noshuf_gain_drop02_f5_300/ripple_0/SubclassSegFormer_MiT-B0_ripple.pth'
    root_path = 'C:/Users/user/PycharmProjects/meta_expand_test/'

    # --- 2. 載入模型物件 ---
    # 若為直接保存模型物件，使用 torch.load 即可重建整個架構與參數
    try:
        checkpoint = torch.load(weight_path, map_location=device)

        if isinstance(checkpoint, torch.nn.Module):
            # 情況 A: checkpoint 本身就是模型物件
            model = checkpoint
        elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            # 情況 B: checkpoint 是字典，且包含 state_dict
            model = SubclassSegFormer(backbone='MiT-B0', num_classes=2, subclass=64).to(device)
            model.load_state_dict(checkpoint['state_dict'])
        else:
            # 情況 C: checkpoint 是字典，但本身就是 state_dict
            model = SubclassSegFormer(backbone='MiT-B0', num_classes=2, subclass=64).to(device)
            model.load_state_dict(checkpoint)

    except Exception as e:
        print(f"模型載入失敗: {e}")
        return

    model.to(device)
    model.eval()

    # --- 3. 遍歷場域執行推論 ---
    for domain_key, label_path in UNSEEN_DOMAINS.items():
        print(f"\n======== 開始場域 {domain_key} 無提示推論 ========")

        test_ds = RippleFeatureDataset(
            root=root_path,
            field=[domain_key],
            unseen_map=UNSEEN_DOMAINS,
            sample_num=20,
            is_pure_test=True,
            list_dir='exclude'
        )

        domain_ious = []

        for idx in tqdm(range(len(test_ds)), desc=f"Domain {domain_key}"):
            img_tensor, gt_tensor = test_ds[idx]
            img_tensor = img_tensor.unsqueeze(0).to(device)
            gt_np = gt_tensor.numpy()

            # --- 無提示推論邏輯 ---
            with torch.no_grad():
                # 原始版本 SubclassSegFormer 回傳 (logits, subclass_map)
                final_output, _ = model(img_tensor)
                pred_mask = torch.argmax(final_output, dim=1).squeeze(0).cpu().numpy()

            # --- 4. 計算 IoU ---
            gt_binary = (gt_np > 0).astype(np.uint8)
            intersection = np.logical_and(pred_mask, gt_binary).sum()
            union = np.logical_or(pred_mask, gt_binary).sum()
            iou = intersection / union if union > 0 else 0.0
            domain_ious.append(iou)

        # --- 5. 儲存結果 ---
        save_name = f'base_npy/iou_results_noprompt_domain_{domain_key}.base_npy'
        np.save(save_name, np.array(domain_ious))
        print(f"✅ 場域 {domain_key} 完成，平均 IoU: {np.mean(domain_ious):.4f}")

if __name__ == "__main__":
    run_unseen_inference_no_prompt()

## SubclassSegformer Point prompt

In [ ]:
import os
import torch
import numpy as np
import random
from tqdm import tqdm
from dataset import RippleFeatureDataset
from semseg.models.subclass_segformer import SubclassSegFormer
from util.analysis_algs import *
from util.features_core import * # 包含 get_R_basis, get_ripple_data_paths 等

def fix_seeds(seed: int = 3407) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

def run_unseen_inference_with_point_prompt():
    fix_seeds(3402)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- 1. 配置與路徑 ---
    UNSEEN_DOMAINS = {
        'F': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG6235/label',
        'G': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG5433/label',
        'H': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201022/label',
        'I': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201211/label',
        'J': 'C:/Users/user/PycharmProjects/meta_expand_test/stereo0420/label'
    }

    # 訓練集路徑（用於提取 Ripple Basis）
    train_fields = 'barrel,sea,LNG,seabass_hmh,noon_jsj'.split(',')
    train_root = r"C:\Users\user\PycharmProjects\organized_ripple_4fold"
    weight_path = '../output_mix5_sb2L2AW_new64_aug10_noshuf_gain_drop02_f5_300/ripple_0/SubclassSegFormer_MiT-B0_ripple.pth'
    root_path = 'C:/Users/user/PycharmProjects/meta_expand_test/'

    # --- 2. 載入模型 ---
    try:
        checkpoint = torch.load(weight_path, map_location=device, weights_only=False)
        if isinstance(checkpoint, torch.nn.Module):
            model = checkpoint
        else:
            model = SubclassSegFormer(backbone='MiT-B0', num_classes=2, subclass=64).to(device)
            model.load_state_dict(checkpoint.get('state_dict', checkpoint))
    except Exception as e:
        print(f"模型載入失敗: {e}")
        return
    model.eval()

    # --- 3. 提取訓練集特徵基底 (Ripple Basis) ---
    # 此步驟模擬學長做法：從已知場域提取子類別空間的分布特徵
    print("正在從訓練集提取 Ripple Basis...")
    training_data = get_ripple_data_paths(train_fields, train_root, k=1)
    r_basis, r_center = get_R_basis(model, training_data, totalsubclass=64, device=device, ver='total_points')
    r_basis = torch.tensor(r_basis).to(device) if not isinstance(r_basis, torch.Tensor) else r_basis.to(device)
    r_center = torch.tensor(r_center).to(device) if not isinstance(r_center, torch.Tensor) else r_center.to(device)

    # --- 4. 遍歷場域執行推論 ---
    for domain_key, label_path in UNSEEN_DOMAINS.items():
        print(f"\n======== 開始場域 {domain_key} 點提示優化推論 ========")

        test_ds = RippleFeatureDataset(
            root=root_path,
            field=[domain_key],
            unseen_map=UNSEEN_DOMAINS,
            sample_num=20,
            is_pure_test=True,
            list_dir='exclude'
        )

        domain_ious = []

        for idx in tqdm(range(len(test_ds)), desc=f"Domain {domain_key}"):
            img_tensor, gt_tensor = test_ds[idx]
            img_tensor = img_tensor.unsqueeze(0).to(device)
            gt_np = gt_tensor.numpy()

            # A. 模擬點擊：獲取正樣本點座標
            center = find_deepest_point(gt_np)
            if center is None: continue

            # B. 提取初始子類別特徵圖
            with torch.no_grad():
                _, subclass_map = model(img_tensor) # subclass_map: (1, 64, H/4, W/4)

            # C. 局部特徵提取與 MIL 優化
            # 將座標映射到特徵圖尺寸 (假設 1/4)
            feat_center = (center[0] // 4, center[1] // 4)
            subclass_np = subclass_map.squeeze(0).cpu().detach().numpy()

            # 提取點擊位置附近的子類別激活 (點提示局部資訊)
            nearby_subclass = nearby_points(subclass_np, feat_center, kernel_size=7)
            nearby_instance = nearby_subclass.reshape(nearby_subclass.shape[0], -1).T

            # 使用學長的 MIL 算法進行子類別權重優化
            # MultiInstanceLearningL2_Obj_L2 結合了基礎基底 (r_basis) 與當前提示資訊
            optimized_weight = MultiInstanceLearningL2_Obj_L2(
                torch.tensor(nearby_instance),
                None,
                r_basis.cpu(), # 根據算法需求決定放置於 CPU 或 GPU
                r_center.cpu(),
                solver='scipy',
                threshold=0.01,
                loop=100
            )

            # D. 使用優化後的權重執行最終推論
            with torch.no_grad():
                subclass_weight = torch.tensor(optimized_weight).to(device).float()
                final_output, _ = model(img_tensor, subclass_weight=subclass_weight)
                pred_mask = torch.argmax(final_output, dim=1).squeeze(0).cpu().numpy()

            # --- 5. 計算 IoU ---
            gt_binary = (gt_np > 0).astype(np.uint8)
            intersection = np.logical_and(pred_mask, gt_binary).sum()
            union = np.logical_or(pred_mask, gt_binary).sum()
            iou = intersection / union if union > 0 else 0.0
            domain_ious.append(iou)

        # --- 6. 儲存結果 ---
        save_name = f'base_npy/iou_results_point_prompt_domain_{domain_key}.base_npy'
        np.save(save_name, np.array(domain_ious))
        print(f"✅ 場域 {domain_key} 完成，平均 IoU: {np.mean(domain_ious):.4f}")

if __name__ == "__main__":
    run_unseen_inference_with_point_prompt()

## SubclassSegformer Unified

In [ ]:
import os
import torch
import numpy as np
from tqdm import tqdm
from dataset import RippleFeatureDataset
from semseg.models.subclass_segformer_unified import SubclassSegFormer_Unified
from util.analysis_algs import *

def run_unseen_inference_unified_noprompt_fixed():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- 1. 配置與路徑 ---
    UNSEEN_DOMAINS = {
        'F': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG6235/label',
        'G': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG5433/label',
        'H': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201022/label',
        'I': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201211/label',
        'J': 'C:/Users/user/PycharmProjects/meta_expand_test/stereo0420/label'
    }

    weight_path = '../output_mix5_SubclassSegFormer_Unified/SubclassSegFormer_Unified_MiT-B0_ripple.pth'
    root_path = 'C:/Users/user/PycharmProjects/meta_expand_test/'

    # --- 2. 載入模型 ---
    model = SubclassSegFormer_Unified(
        backbone='MiT-B0',
        num_classes=2,
        subclass=64,
        num_prompts=5
    ).to(device)

    checkpoint = torch.load(weight_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint.get('state_dict', checkpoint), strict=False)
    model.eval()

    # --- 3. 遍歷場域執行推論 ---
    for domain_key, label_path in UNSEEN_DOMAINS.items():
        print(f"\n======== 開始場域 {domain_key} [unified No-Prompt] 推論 ========")

        test_ds = RippleFeatureDataset(
            root=root_path,
            field=[domain_key],
            unseen_map=UNSEEN_DOMAINS,
            sample_num=20,
            is_pure_test=True,
            list_dir='exclude'
        )

        domain_ious = []

        for idx in tqdm(range(len(test_ds)), desc=f"Domain {domain_key}"):
            img_tensor, gt_tensor = test_ds[idx]
            img_tensor = img_tensor.unsqueeze(0).to(device)
            gt_np = gt_tensor.numpy()

            # --- 核心修正點 ---
            with torch.no_grad():
                # 強制開啟 output_subclass=True，以避免底層 SubclassBlock 回傳值數量不足的問題
                # 此時 model 會回傳 (final_output, subclass_map)
                final_output, _ = model(img_tensor, subclass_weight=None, output_subclass=True)

                # 取得預測結果 (B, C, H, W) -> (H, W)
                pred_mask = torch.argmax(final_output, dim=1).squeeze(0).cpu().numpy()

            # --- 4. 計算 IoU ---
            gt_binary = (gt_np > 0).astype(np.uint8)
            intersection = np.logical_and(pred_mask, gt_binary).sum()
            union = np.logical_or(pred_mask, gt_binary).sum()
            iou = intersection / union if union > 0 else 0.0
            domain_ious.append(iou)

        # --- 5. 儲存結果 ---
        save_name = f'iou_results_unified_noprompt_domain_{domain_key}.base_npy'
        np.save(save_name, np.array(domain_ious))
        print(f"✅ 場域 {domain_key} 完成，平均 IoU: {np.mean(domain_ious):.4f}")

if __name__ == "__main__":
    run_unseen_inference_unified_noprompt_fixed()

## SubclassSegformer Unified mil tuning Point prompt

In [ ]:
import os
import torch
import numpy as np
from tqdm import tqdm
from dataset import RippleFeatureDataset
from semseg.models.subclass_segformer_unified import SubclassSegFormer_Unified
from util.analysis_algs import *

def run_unseen_inference_unified_mil():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- 1. 配置與路徑 ---
    UNSEEN_DOMAINS = {
        'F': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG6235/label',
        'G': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG5433/label',
        'H': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201022/label',
        'I': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201211/label',
        'J': 'C:/Users/user/PycharmProjects/meta_expand_test/stereo0420/label'
    }

    # 使用 unified 版本的權重
    weight_path = '../output_mix5_SubclassSegFormer_Unified/SubclassSegFormer_Unified_MiT-B0_ripple.pth'
    root_path = 'C:/Users/user/PycharmProjects/meta_expand_test/'

    # --- 2. 載入模型 ---
    # 設定 num_prompts=5 以符合 unified 架構設計
    model = SubclassSegFormer_Unified(
        backbone='MiT-B0',
        num_classes=2,
        subclass=64,
        num_prompts=5
    ).to(device)

    checkpoint = torch.load(weight_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint.get('state_dict', checkpoint), strict=False)
    model.eval()

    # --- 3. 遍歷場域執行推論 ---
    for domain_key, label_path in UNSEEN_DOMAINS.items():
        print(f"\n======== 開始場域 {domain_key} [unified MIL] 推論 ========")

        test_ds = RippleFeatureDataset(
            root=root_path,
            field=[domain_key],
            unseen_map=UNSEEN_DOMAINS,
            sample_num=20,
            is_pure_test=True,
            list_dir='exclude'
        )

        domain_ious = []

        for idx in tqdm(range(len(test_ds)), desc=f"Domain {domain_key}"):
            img_tensor, gt_tensor = test_ds[idx]
            img_tensor = img_tensor.unsqueeze(0).to(device)
            gt_np = gt_tensor.numpy()

            # A. 模擬點擊：中心點提取
            center = find_deepest_point(gt_np)
            if center is None: continue

            # 座標映射 (1/4 尺寸)
            pos_coords = torch.tensor([[center[0] // 4, center[1] // 4]]).long().to(device)

            # B. 提取特徵與執行最新 Logistic MIL 優化
            with torch.no_grad():
                # 使用 output_subclass=True 獲取激活圖
                _, subclass_map = model(img_tensor, output_subclass=True)

            # 參數使用：500 圈, Kernel 9, Exclusion 15 (最新調校參數)
            mil_params = MultiInstanceLearning_Logistic_Tuning(
                subclass_map=subclass_map,
                pos_coords=pos_coords,
                iterations=500,
                pos_kernel=9,
                exclusion_radius=15,
                neg_ratio=5,
                device=device
            )

            # C. 傳回優化權重執行最終推論
            with torch.no_grad():
                final_output = model(img_tensor, subclass_weight=mil_params)
                pred_mask = torch.argmax(final_output, dim=1).squeeze(0).cpu().numpy()

            # D. 計算 IoU
            gt_binary = (gt_np > 0).astype(np.uint8)
            intersection = np.logical_and(pred_mask, gt_binary).sum()
            union = np.logical_or(pred_mask, gt_binary).sum()
            iou = intersection / union if union > 0 else 0.0
            domain_ious.append(iou)

        # --- 4. 儲存結果（修改檔名以辨識版本） ---
        save_name = f'base_npy/iou_results_unified_mil_v2_domain_{domain_key}.base_npy'
        np.save(save_name, np.array(domain_ious))
        print(f"✅ 場域 {domain_key} 完成，平均 IoU: {np.mean(domain_ious):.4f}，存檔：{save_name}")

if __name__ == "__main__":
    run_unseen_inference_unified_mil()

## SubclassSegformer Unified mil v2 Point prompt

In [3]:
import matplotlib.patches as patches

def plot_top_bottom_results(results_list, domain_key, save_dir='visualization_results'):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # 排序邏輯不變
    results_sorted = sorted(results_list, key=lambda x: x['iou'])
    bottom_5 = results_sorted[:3]
    top_5 = results_sorted[-3:][::-1]

    to_plot = [('Top', top_5), ('Bottom', bottom_5)]

    for label, samples in to_plot:
        fig, axes = plt.subplots(len(samples), 3, figsize=(18, 5 * len(samples)))
        fig.suptitle(f"Domain {domain_key} - {label} 5 IoU (Corrected RoI)", fontsize=20)

        for i, res in enumerate(samples):
            # A. 原圖還原 (與單元測試一致)
            img = res['image'].transpose(1, 2, 0)
            img = (img - img.min()) / (img.max() - img.min() + 1e-5)
            h_img, w_img, _ = img.shape

            # B. Ground Truth (綠色半透明，與單元測試風格一致)
            gt = res['gt']
            gt_masked = np.ma.masked_where(gt == 0, gt)

            # C. Prediction Overlay
            pred = res['pred']
            overlay = img.copy()
            overlay[pred > 0] = overlay[pred > 0] * 0.6 + np.array([1, 0, 0]) * 0.4

            # 繪圖
            axes[i, 0].imshow(img)
            axes[i, 1].imshow(img) # 背景用原圖
            axes[i, 1].imshow(gt_masked, cmap='Greens', alpha=0.5) # 疊加綠色 GT
            axes[i, 2].imshow(overlay)

            # --- 修正後的 RoI 繪製邏輯 ---
            if 'roi_box' in res and res['roi_box'] is not None:
                # 嚴格對照您的解包方式：y_min, y_max, x_min, x_max
                y_min, y_max, x_min, x_max = res['roi_box']

                # 計算縮放率 (假設特徵圖是 200x200，影像實際大小可能是 800x800)
                # 您單元測試中的 scale_factor 是 4，這裡改為動態計算更為嚴謹
                scale_h, scale_w = h_img / 200.0, w_img / 200.0

                rect_w = (x_max - x_min) * scale_w
                rect_h = (y_max - y_min) * scale_h

                # patches.Rectangle((x_min, y_min), width, height)
                rect_edge = patches.Rectangle(
                    (x_min * scale_w, y_min * scale_h),
                    rect_w,
                    rect_h,
                    linewidth=2,
                    edgecolor='yellow',
                    facecolor='none',
                    linestyle='--',
                    zorder=10
                )
                axes[i, 2].add_patch(rect_edge)
                axes[i, 2].set_title(f"IoU: {res['iou']:.4f}")

            for ax in axes[i]:
                ax.axis('off')

        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"domain_{domain_key}_{label}_corrected.png"), dpi=150)
        plt.close()

In [2]:
import os
import torch
import numpy as np
import random
from tqdm import tqdm
from dataset import RippleFeatureDataset
from semseg.models.subclass_segformer_unified import SubclassSegFormer_Unified
from util.analysis_algs import *
from util.features_core import * # 包含 get_R_basis, get_ripple_data_paths 等

def fix_seeds(seed: int = 3407) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

def run_unseen_inference_with_point_prompt_v3():
    fix_seeds(3402)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- 1. 配置與路徑 ---
    UNSEEN_DOMAINS = {
        'F': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG6235/label',
        'G': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG5433/label',
        'H': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201022/label',
        'I': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201211/label',
        'J': 'C:/Users/user/PycharmProjects/meta_expand_test/stereo0420/label'
    }

    # 訓練集路徑（用於提取 Ripple Basis）
    train_fields = 'barrel,sea,LNG,seabass_hmh,noon_jsj'.split(',')
    train_root = r"C:\Users\user\PycharmProjects\organized_ripple_4fold"
    weight_path = '../output_mix5_SubclassSegFormer_Unified/SubclassSegFormer_Unified_MiT-B0_ripple.pth'
    root_path = 'C:/Users/user/PycharmProjects/meta_expand_test/'
    basis_path = 'r_basis.pt'
    center_path = 'r_center.pt'

    # --- 2. 載入模型 ---
    try:
        checkpoint = torch.load(weight_path, map_location=device, weights_only=False)
        if isinstance(checkpoint, torch.nn.Module):
            model = checkpoint
        else:
            model = SubclassSegFormer_Unified(backbone='MiT-B0', num_classes=2, subclass=64,num_prompts=5).to(device)
            model.load_state_dict(checkpoint.get('state_dict', checkpoint))
    except Exception as e:
        print(f"模型載入失敗: {e}")
        return
    model.eval()


    if os.path.exists(basis_path) and os.path.exists(center_path):
        # 情況 A: 檔案存在，直接載入
        print(f">>> 檢測到快取檔案，正在載入 Ripple Basis...")
        r_basis = torch.load(basis_path).to(device)
        r_center = torch.load(center_path).to(device)
    else:
        # 情況 B: 檔案不存在，執行提取流程
        print(">>> 未發現快取，正在從訓練集提取 Ripple Basis (這可能需要一點時間)...")
        training_data = get_ripple_data_paths(train_fields, train_root, k=1)

        # 執行提取 (ver='total_points')
        r_basis_np, r_center_np = get_R_basis(model, training_data, totalsubclass=64, device=device, ver='total_points')

        # 轉換為 Tensor 並確保格式正確
        r_basis = torch.from_numpy(r_basis_np).float().to(device)
        r_center = torch.from_numpy(r_center_np).float().to(device)

        # 儲存到本地，下次就不用再跑了
        torch.save(r_basis, basis_path)
        torch.save(r_center, center_path)
        print(f"✅ Basis 提取完成並已存檔至 {basis_path}")

    # --- 4. 遍歷場域執行推論 ---
    for domain_key, label_path in UNSEEN_DOMAINS.items():
        print(f"\n======== 開始場域 {domain_key} 點提示優化推論 ========")

        test_ds = RippleFeatureDataset(
            root=root_path,
            field=[domain_key],
            unseen_map=UNSEEN_DOMAINS,
            sample_num=20,
            is_pure_test=True,
            list_dir='exclude'
        )

        domain_ious = []
        domain_results_cache = [] # 新增：用於快取該場域的所有結果

        for idx in tqdm(range(len(test_ds)), desc=f"Domain {domain_key}"):
            img_tensor, gt_tensor = test_ds[idx]
            img_tensor = img_tensor.unsqueeze(0).to(device)
            gt_np = gt_tensor.numpy()

            # C. 局部特徵提取與 MIL 優化
            mask_200 = F.interpolate(gt_tensor.unsqueeze(0).unsqueeze(0).float(),
                                     size=(200, 200), mode='nearest').squeeze().numpy().astype(np.uint8)

            _, roi_box, _ = find_roi_by_bbox(mask_200, margin_px=5)

            # B. 提取初始子類別特徵圖
            with torch.no_grad():
                _, subclass_map = model(img_tensor) # subclass_map: (1, 64, H/4, W/4)

            # 提取點擊位置附近的子類別激活 (點提示局部資訊)
            p_inst, n_inst = get_prompt_instances(subclass_map, roi_box, neg_ratio=1.0)

            # 使用學長的 MIL 算法進行子類別權重優化
            # MultiInstanceLearningL2_Obj_L2 結合了基礎基底 (r_basis) 與當前提示資訊
            optimized_weight = MultiInstanceLearningL2_Obj_L2(
                p_inst.cpu(),     # 正樣本矩陣
                n_inst.cpu(),     # 負樣本矩陣 (新加入)
                r_basis.cpu(),
                r_center.cpu(),
                solver='scipy',
                threshold=0.01,
                loop=100
            )

            # D. 使用優化後的權重執行最終推論
            with torch.no_grad():
                subclass_weight = torch.tensor(optimized_weight).to(device).float()
                final_output, _ = model(img_tensor, subclass_weight=subclass_weight)
                pred_mask = torch.argmax(final_output, dim=1).squeeze(0).cpu().numpy()

            # --- 5. 計算 IoU ---
            gt_binary = (gt_np > 0).astype(np.uint8)
            intersection = np.logical_and(pred_mask, gt_binary).sum()
            union = np.logical_or(pred_mask, gt_binary).sum()
            iou = intersection / union if union > 0 else 0.0
            domain_ious.append(iou)
            domain_results_cache.append({
            'image': img_tensor.squeeze(0).cpu().numpy(),
            'gt': gt_binary,
            'pred': pred_mask,
            'iou': iou,
            'roi_box': roi_box
            })

        # --- 6. 儲存結果 ---
        #plot_top_bottom_results(domain_results_cache, domain_key)
        save_name = f'base_npy/iou_results_unified_mil_v3_domain_{domain_key}.base_npy'
        np.save(save_name, np.array(domain_ious))
        print(f"✅ 場域 {domain_key} 完成，平均 IoU: {np.mean(domain_ious):.4f}")

if __name__ == "__main__":
    run_unseen_inference_with_point_prompt_v3()

C:\Users\user\AppData\Local\Temp\ipykernel_16112\4211503315.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  r_basis = torch.load(basis_path).to(device)
C:\Users\user\Ap

>>> 檢測到快取檔案，正在載入 Ripple Basis...

======== 開始場域 F 點提示優化推論 ========
>>> [Standardized Test] F: 讀取清單排除 20 張，剩餘 162 張作為測試集。


Domain F:   0%|          | 0/162 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_16112\4211503315.py:122: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  subclass_weight = torch.tensor(optimized_weight).to(device).float()
Domain F: 100%|██████████| 162/162 [00:04<00:00, 34.77it/s]


✅ 場域 F 完成，平均 IoU: 0.0212

======== 開始場域 G 點提示優化推論 ========
>>> [Standardized Test] G: 讀取清單排除 20 張，剩餘 87 張作為測試集。


Domain G: 100%|██████████| 87/87 [00:02<00:00, 38.24it/s]


...

✅ 場域 H 完成，平均 IoU: 0.0067

======== 開始場域 I 點提示優化推論 ========
>>> [Standardized Test] I: 讀取清單排除 20 張，剩餘 98 張作為測試集。


Domain I: 100%|██████████| 98/98 [00:02<00:00, 38.10it/s]


✅ 場域 I 完成，平均 IoU: 0.0001

======== 開始場域 J 點提示優化推論 ========
>>> [Standardized Test] J: 讀取清單排除 20 張，剩餘 140 張作為測試集。


Domain J: 100%|██████████| 140/140 [00:03<00:00, 35.79it/s]

✅ 場域 J 完成，平均 IoU: 0.0028


# Segformer B1 F-J

In [ ]:
import os
import torch
import numpy as np
from tqdm import tqdm
from pathlib import Path
import torch.nn.functional as F

# 引入你的 Dataset (請確保 RippleFeatureDataset 在 is_pure_test=True 時是回傳 RGB 原圖而非特徵檔)
from dataset import RippleFeatureDataset
# 引入純淨版 Segformer
from semseg.models.segformer import SegFormer

def run_5fold_segformer_b1_inference():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- 1. 配置與路徑 ---
    UNSEEN_DOMAINS = {
        'F': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG6235/label',
        'G': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG5433/label',
        'H': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201022/label',
        'I': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201211/label',
        'J': 'C:/Users/user/PycharmProjects/meta_expand_test/stereo0420/label'
    }

    root_path = 'C:/Users/user/PycharmProjects/meta_expand_test/'
    weight_path = r'C:\Users\user\PycharmProjects\subclass_segformer\output_mix5_Segformer_B1\ripple_0\Segformer_MiT-B1.pth'

    # 5-Fold 相關設定
    FOLDS_NUM = 5
    FOLDS_BASE_DIR = Path("folds_experiment")
    OUTPUT_DIR = Path("all_npy")  # 統一輸出路徑
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    domains = list(UNSEEN_DOMAINS.keys())

    # 分別紀錄跨 Fold 的 IoU 與 Precision 結果
    all_results_iou = {d: [] for d in domains}
    all_results_prec = {d: [] for d in domains}

    # --- 2. 初始化模型與載入權重 ---
    print(f"📦 正在載入模型權重: {weight_path}")
    try:
        # 初始化純淨版 Segformer B1 (假設分類數 num_classes 為 2)
        model = SegFormer(backbone='MiT-B1', num_classes=2)

        # 載入 state_dict
        checkpoint = torch.load(weight_path, map_location=device)
        model.load_state_dict(checkpoint, strict=True)
        print("✅ 模型權重載入成功！")

    except Exception as e:
        print(f"❌ 模型載入失敗: {e}")
        return

    model.to(device)
    model.eval()

    # --- 3. 5-Fold 迴圈推論 ---
    for fold_idx in range(FOLDS_NUM):
        fold_key = f"fold_{fold_idx}"
        fold_dir = FOLDS_BASE_DIR / fold_key

        print(f"\n{'='*70}")
        print(f"🚀 啟動 Segformer-B1 推論 | Round: {fold_idx + 1}/{FOLDS_NUM} ({fold_key})")
        print(f"{'='*70}")

        for domain_key in domains:
            # 透過 list_dir 排除該 Fold 的 5 張提示圖
            test_ds = RippleFeatureDataset(
                root=root_path,
                field=[domain_key],
                unseen_map=UNSEEN_DOMAINS,
                sample_num=5,  # 設定為排除 5 張
                is_pure_test=True,
                list_dir=str(fold_dir)
            )

            domain_ious = []
            domain_precs = []

            for idx in tqdm(range(len(test_ds)), desc=f"Evaluating {domain_key} (Fold {fold_idx})"):
                img_tensor, gt_tensor = test_ds[idx]
                img_tensor = img_tensor.unsqueeze(0).to(device) # (1, 3, H, W)
                gt_np = gt_tensor.numpy()                       # (H, W)

                # --- Baseline 推論邏輯 ---
                with torch.no_grad():
                    output = model(img_tensor)

                    # 防呆解包
                    if isinstance(output, tuple):
                        logits = output[0]
                    else:
                        logits = output

                    # 【保險機制】確保預測結果的大小與 Ground Truth 一致
                    if logits.shape[2:] != gt_tensor.shape:
                        logits = F.interpolate(logits, size=gt_tensor.shape, mode='bilinear', align_corners=False)

                    # 取得預測的 Mask (1, H, W) -> (H, W)
                    pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

                # --- 計算 IoU 與 Precision ---
                gt_binary = (gt_np > 0).astype(np.uint8)
                pred_binary = (pred_mask == 1).astype(np.uint8)

                tp = np.logical_and(pred_binary == 1, gt_binary == 1).sum()
                fp = np.logical_and(pred_binary == 1, gt_binary == 0).sum()
                union = np.logical_or(pred_binary == 1, gt_binary == 1).sum()

                # 計算 IoU
                iou = 1.0 if union == 0 else (tp / union)
                domain_ious.append(iou)

                # 計算 Precision (TP / (TP + FP))
                prec = tp / (tp + fp + 1e-6)
                domain_precs.append(prec)

            mean_iou = np.mean(domain_ious)
            mean_prec = np.mean(domain_precs)

            all_results_iou[domain_key].append(mean_iou)
            all_results_prec[domain_key].append(mean_prec)

            # --- 4. 儲存結果 (堆疊為 (N, 2) 並存至 all_npy) ---
            results_array = np.stack((domain_ious, domain_precs), axis=-1)
            npy_path = OUTPUT_DIR / f'metrics_segformerB1_fold{fold_idx}_domain_{domain_key}.base_npy'
            np.save(npy_path, results_array)

            print(f"   ▶ {domain_key} 測試完成 | Mean IoU: {mean_iou:.4f} | Mean Prec: {mean_prec:.4f}")

    # --- 5. 總結報告 ---
    print("\n" + "=" * 60)
    print("📊 Segformer-B1 5-Fold 實驗總結報告")
    print("=" * 60)
    for d in domains:
        mean_across_folds_iou = np.mean(all_results_iou[d])
        mean_across_folds_prec = np.mean(all_results_prec[d])
        print(f"Domain {d} - 5 Folds Mean IoU: {mean_across_folds_iou:.4f} | Mean Prec: {mean_across_folds_prec:.4f}")
        for idx in range(FOLDS_NUM):
            print(f"   - Fold {idx} -> IoU: {all_results_iou[d][idx]:.4f}, Prec: {all_results_prec[d][idx]:.4f}")

if __name__ == "__main__":
    run_5fold_segformer_b1_inference()

# Segformer B0 F-J

In [ ]:
import os
import torch
import numpy as np
from tqdm import tqdm
from pathlib import Path
import torch.nn.functional as F

# 引入你的 Dataset (請確保 RippleFeatureDataset 在 is_pure_test=True 時是回傳 RGB 原圖而非特徵檔)
from dataset import RippleFeatureDataset
# 引入純淨版 Segformer
from semseg.models.segformer import SegFormer

def run_5fold_segformer_b0_inference():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- 1. 配置與路徑 ---
    UNSEEN_DOMAINS = {
        'F': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG6235/label',
        'G': 'C:/Users/user/PycharmProjects/meta_expand_test/IMG5433/label',
        'H': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201022/label',
        'I': 'C:/Users/user/PycharmProjects/meta_expand_test/sea20201211/label',
        'J': 'C:/Users/user/PycharmProjects/meta_expand_test/stereo0420/label'
    }

    root_path = 'C:/Users/user/PycharmProjects/meta_expand_test/'
    weight_path = r'C:\Users\user\PycharmProjects\subclass_segformer\output_mix5_Segformer_B0\ripple_0\Segformer_MiT-B0.pth'

    # 5-Fold 相關設定
    FOLDS_NUM = 5
    FOLDS_BASE_DIR = Path("folds_experiment")
    OUTPUT_DIR = Path("all_npy")  # 統一輸出路徑
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    domains = list(UNSEEN_DOMAINS.keys())

    # 分別紀錄跨 Fold 的 IoU 與 Precision 結果
    all_results_iou = {d: [] for d in domains}
    all_results_prec = {d: [] for d in domains}

    # --- 2. 初始化模型與載入權重 ---
    print(f"📦 正在載入模型權重: {weight_path}")
    try:
        # 初始化純淨版 Segformer B0 (假設分類數 num_classes 為 2)
        model = SegFormer(backbone='MiT-B0', num_classes=2)

        # 載入 state_dict
        checkpoint = torch.load(weight_path, map_location=device)
        model.load_state_dict(checkpoint, strict=True)
        print("✅ 模型權重載入成功！")

    except Exception as e:
        print(f"❌ 模型載入失敗: {e}")
        return

    model.to(device)
    model.eval()

    # --- 3. 5-Fold 迴圈推論 ---
    for fold_idx in range(FOLDS_NUM):
        fold_key = f"fold_{fold_idx}"
        fold_dir = FOLDS_BASE_DIR / fold_key

        print(f"\n{'='*70}")
        print(f"🚀 啟動 Segformer-B0 推論 | Round: {fold_idx + 1}/{FOLDS_NUM} ({fold_key})")
        print(f"{'='*70}")

        for domain_key in domains:
            # 透過 list_dir 排除該 Fold 的 5 張提示圖
            test_ds = RippleFeatureDataset(
                root=root_path,
                field=[domain_key],
                unseen_map=UNSEEN_DOMAINS,
                sample_num=5,  # 設定為排除 5 張
                is_pure_test=True,
                list_dir=str(fold_dir)
            )

            domain_ious = []
            domain_precs = []

            for idx in tqdm(range(len(test_ds)), desc=f"Evaluating {domain_key} (Fold {fold_idx})"):
                img_tensor, gt_tensor = test_ds[idx]
                img_tensor = img_tensor.unsqueeze(0).to(device) # (1, 3, H, W)
                gt_np = gt_tensor.numpy()                       # (H, W)

                # --- Baseline 推論邏輯 ---
                with torch.no_grad():
                    output = model(img_tensor)

                    # 防呆解包
                    if isinstance(output, tuple):
                        logits = output[0]
                    else:
                        logits = output

                    # 【保險機制】確保預測結果的大小與 Ground Truth 一致
                    if logits.shape[2:] != gt_tensor.shape:
                        logits = F.interpolate(logits, size=gt_tensor.shape, mode='bilinear', align_corners=False)

                    # 取得預測的 Mask (1, H, W) -> (H, W)
                    pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

                # --- 計算 IoU 與 Precision ---
                gt_binary = (gt_np > 0).astype(np.uint8)
                pred_binary = (pred_mask == 1).astype(np.uint8)

                tp = np.logical_and(pred_binary == 1, gt_binary == 1).sum()
                fp = np.logical_and(pred_binary == 1, gt_binary == 0).sum()
                union = np.logical_or(pred_binary == 1, gt_binary == 1).sum()

                # 計算 IoU
                iou = 1.0 if union == 0 else (tp / union)
                domain_ious.append(iou)

                # 計算 Precision (TP / (TP + FP))
                prec = tp / (tp + fp + 1e-6)
                domain_precs.append(prec)

            mean_iou = np.mean(domain_ious)
            mean_prec = np.mean(domain_precs)

            all_results_iou[domain_key].append(mean_iou)
            all_results_prec[domain_key].append(mean_prec)

            # --- 4. 儲存結果 (堆疊為 (N, 2) 並存至 all_npy) ---
            results_array = np.stack((domain_ious, domain_precs), axis=-1)
            npy_path = OUTPUT_DIR / f'metrics_segformerB0_fold{fold_idx}_domain_{domain_key}.base_npy'
            np.save(npy_path, results_array)

            print(f"   ▶ {domain_key} 測試完成 | Mean IoU: {mean_iou:.4f} | Mean Prec: {mean_prec:.4f}")

    # --- 5. 總結報告 ---
    print("\n" + "=" * 60)
    print("📊 Segformer-B0 5-Fold 實驗總結報告")
    print("=" * 60)
    for d in domains:
        mean_across_folds_iou = np.mean(all_results_iou[d])
        mean_across_folds_prec = np.mean(all_results_prec[d])
        print(f"Domain {d} - 5 Folds Mean IoU: {mean_across_folds_iou:.4f} | Mean Prec: {mean_across_folds_prec:.4f}")
        for idx in range(FOLDS_NUM):
            print(f"   - Fold {idx} -> IoU: {all_results_iou[d][idx]:.4f}, Prec: {all_results_prec[d][idx]:.4f}")

if __name__ == "__main__":
    run_5fold_segformer_b0_inference()

# A-E

In [ ]:
import itertools
import numpy as np
import os
import torch
from glob import glob
from torch import Tensor
from torch.utils.data import Dataset
from torchvision import io
from typing import List, Tuple, Union
from pathlib import Path
import random
import torchvision.transforms.functional as TF

class RippleFeatureDataset(Dataset):
    """
    專為 A-E 已見場域設計的 Dataset。
    直接讀取特定場域底下的 images 與 labels_detectron2 資料夾 (完整 500 張)。
    """
    def __init__(self, root: str, field: list = None, split: str = 'train', sample_num=20,
                 val_size=5, seed=3402, is_pure_test=False, list_dir: str = None):
        super().__init__()
        self.root = root
        self.split = split
        self.ignore_label = 255
        self.n_classes = 2
        self.files = []
        self.sample_num = sample_num
        self.val_size = val_size
        self.is_pure_test = is_pure_test
        self.list_dir = list_dir

        rng = random.Random(seed)

        if field is None:
            raise ValueError("❌ 錯誤: 必須指定 field 參數。")

        for f in field:
            # 依據指定結構：直接進入場域資料夾下的 images 與 labels_detectron2
            img_dir = Path(self.root) / f / 'images'
            lbl_dir = Path(self.root) / f / 'labels_detectron2'

            # 支援多種影像格式
            img_files = []
            for ext in ['*.png', '*.jpg', '*.JPG', '*.jpeg']:
                img_files.extend(list(img_dir.glob(ext)))
            img_files = sorted(img_files)

            full_paired_pool = []
            for img_path in img_files:
                # 根據影像名稱配對標籤
                lbl_path = lbl_dir / f"{img_path.stem}.png"
                if lbl_path.exists():
                    full_paired_pool.append((str(img_path), str(lbl_path)))

            # 排除與切分邏輯
            if self.is_pure_test and self.list_dir:
                exclude_names = self._load_exclude_names(f)
                current_test_files = [p for p in full_paired_pool if os.path.basename(p[0]) not in exclude_names]
                self.files.extend(current_test_files)
            else:
                rng.shuffle(full_paired_pool)
                few_shot_pool = full_paired_pool[:self.sample_num]
                test_pool = full_paired_pool[self.sample_num:]

                if self.is_pure_test:
                    self.files.extend(test_pool)
                elif self.split == 'val':
                    self.files.extend(few_shot_pool[:self.val_size])
                else:
                    self.files.extend(few_shot_pool[self.val_size:])

        print(f">>> [Dataset A - Eval Mode] {field} ({self.split}): 已切換為讀取原圖，總共 {len(self.files)} 張。")

    def _load_exclude_names(self, domain_key):
        """
        從 list_dir 中讀取對應場域的 val 清單，提取所有檔名進行排除。
        """
        exclude_set = set()
        list_file = Path(self.list_dir) / f"{domain_key}_mask_hard_samples_val_list.txt"

        if list_file.exists():
            with open(list_file, 'r', encoding='utf-8') as f:
                for line in f:
                    if "IMG: " in line:
                        name = line.split('|')[0].replace("IMG: ", "").strip()
                        exclude_set.add(name)
        else:
            print(f"⚠️ 警告: 找不到清單檔案 {list_file.name}，請檢查路徑。")

        return exclude_set

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        data_path, lbl_path = self.files[index]

        # 讀取影像與標籤
        img = io.read_image(data_path, io.ImageReadMode.RGB)
        lbl = io.read_image(lbl_path, io.ImageReadMode.GRAY)

        # 影像標準化與縮放處理
        img = TF.resize(img, [800, 800], interpolation=TF.InterpolationMode.BILINEAR)
        img = TF.convert_image_dtype(img, torch.float)
        img = TF.normalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        lbl = TF.resize(lbl, [800, 800], interpolation=TF.InterpolationMode.NEAREST).squeeze()

        return img, lbl.long()

In [ ]:
def run_seen_inference_baseline():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- 1. 配置與 A-E 場域路徑 ---
    ROOT_AE = 'C:/Users/user/PycharmProjects/organized_ripple_4fold'
    FIELDS_AE = ['barrel', 'sea', 'LNG', 'seabass_hmh', 'noon_jsj']

    # 排除清單儲存目錄 (若無請設為 None 或對應路徑)
    LIST_DIR = 'C:/Users/user/PycharmProjects/meta_expand_test/your_list_folder'

    # 權重路徑
    weight_path = r'C:\Users\user\PycharmProjects\subclass_segformer\output_mix5_Segformer_B0\ripple_0\Segformer_MiT-B0.pth'

    # --- 2. 初始化模型與載入權重 ---
    print(f"📦 正在載入模型權重: {weight_path}")
    try:
        model = SegFormer(backbone='MiT-B0', num_classes=2)
        checkpoint = torch.load(weight_path, map_location=device)
        model.load_state_dict(checkpoint, strict=True)
        print("✅ 模型權重載入成功！")
    except Exception as e:
        print(f"❌ 模型載入失敗: {e}")
        return

    model.to(device)
    model.eval()

    # --- 3. 遍歷 A-E 場域執行推論 ---
    for domain_key in FIELDS_AE:
        print(f"\n======== 開始場域 {domain_key} Baseline 推論 ========")

        # 實例化專為 A-E 設計的測試集
        test_ds = RippleFeatureDataset(
            root=ROOT_AE,
            field=[domain_key],
            is_pure_test=True,
            list_dir=LIST_DIR
        )

        domain_ious = []

        for idx in tqdm(range(len(test_ds)), desc=f"Domain {domain_key}"):
            img_tensor, gt_tensor = test_ds[idx]
            img_tensor = img_tensor.unsqueeze(0).to(device) # (1, 3, 800, 800)
            gt_np = gt_tensor.numpy()                       # (800, 800)

            with torch.no_grad():
                output = model(img_tensor)

                if isinstance(output, tuple):
                    logits = output[0]
                else:
                    logits = output

                # 確保預測與 GT 尺寸一致 (Dataset 已內建 800x800，此處防呆)
                if logits.shape[2:] != gt_tensor.shape:
                    logits = F.interpolate(logits, size=gt_tensor.shape, mode='bilinear', align_corners=False)

                pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

            # --- 4. 計算 IoU ---
            gt_binary = (gt_np > 0).astype(np.uint8)
            intersection = np.logical_and(pred_mask, gt_binary).sum()
            union = np.logical_or(pred_mask, gt_binary).sum()
            iou = intersection / union if union > 0 else 0.0
            domain_ious.append(iou)

        # --- 5. 儲存結果 ---
        os.makedirs('base_npy/segformerB0_npy', exist_ok=True)
        save_name = f'base_npy/segformerB0_npy/iou_results_segformerB0_domain_{domain_key}.base_npy'
        np.save(save_name, np.array(domain_ious))
        print(f"🎉 場域 {domain_key} 完成，平均 IoU: {np.mean(domain_ious):.4f}")

if __name__ == "__main__":
    run_seen_inference_baseline()

In [ ]:
def run_seen_inference_baseline():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- 1. 配置與 A-E 場域路徑 ---
    ROOT_AE = 'C:/Users/user/PycharmProjects/organized_ripple_4fold'
    FIELDS_AE = ['barrel', 'sea', 'LNG', 'seabass_hmh', 'noon_jsj']

    # 排除清單儲存目錄 (若無請設為 None 或對應路徑)
    LIST_DIR = 'C:/Users/user/PycharmProjects/meta_expand_test/your_list_folder'

    # 權重路徑
    weight_path = r'C:\Users\user\PycharmProjects\subclass_segformer\output_mix5_Segformer_B1\ripple_0\Segformer_MiT-B1.pth'

    # --- 2. 初始化模型與載入權重 ---
    print(f"📦 正在載入模型權重: {weight_path}")
    try:
        model = SegFormer(backbone='MiT-B1', num_classes=2)
        checkpoint = torch.load(weight_path, map_location=device)
        model.load_state_dict(checkpoint, strict=True)
        print("✅ 模型權重載入成功！")
    except Exception as e:
        print(f"❌ 模型載入失敗: {e}")
        return

    model.to(device)
    model.eval()

    # --- 3. 遍歷 A-E 場域執行推論 ---
    for domain_key in FIELDS_AE:
        print(f"\n======== 開始場域 {domain_key} Baseline 推論 ========")

        # 實例化專為 A-E 設計的測試集
        test_ds = RippleFeatureDataset(
            root=ROOT_AE,
            field=[domain_key],
            is_pure_test=True,
            list_dir=LIST_DIR
        )

        domain_ious = []

        for idx in tqdm(range(len(test_ds)), desc=f"Domain {domain_key}"):
            img_tensor, gt_tensor = test_ds[idx]
            img_tensor = img_tensor.unsqueeze(0).to(device) # (1, 3, 800, 800)
            gt_np = gt_tensor.numpy()                       # (800, 800)

            with torch.no_grad():
                output = model(img_tensor)

                if isinstance(output, tuple):
                    logits = output[0]
                else:
                    logits = output

                # 確保預測與 GT 尺寸一致 (Dataset 已內建 800x800，此處防呆)
                if logits.shape[2:] != gt_tensor.shape:
                    logits = F.interpolate(logits, size=gt_tensor.shape, mode='bilinear', align_corners=False)

                pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

            # --- 4. 計算 IoU ---
            gt_binary = (gt_np > 0).astype(np.uint8)
            intersection = np.logical_and(pred_mask, gt_binary).sum()
            union = np.logical_or(pred_mask, gt_binary).sum()
            iou = intersection / union if union > 0 else 0.0
            domain_ious.append(iou)

        # --- 5. 儲存結果 ---
        os.makedirs('base_npy/segformerB1_npy', exist_ok=True)
        save_name = f'base_npy/segformerB1_npy/iou_results_segformerB1_domain_{domain_key}.base_npy'
        np.save(save_name, np.array(domain_ious))
        print(f"🎉 場域 {domain_key} 完成，平均 IoU: {np.mean(domain_ious):.4f}")

if __name__ == "__main__":
    run_seen_inference_baseline()

# SubclassSegformer Unified A-E

In [ ]:
from semseg.models.subclass_segformer_unified import SubclassSegFormer_Unified
from util.analysis_algs import *
def run_seen_inference_baseline():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # --- 1. 配置與 A-E 場域路徑 ---
    ROOT_AE = 'C:/Users/user/PycharmProjects/organized_ripple_4fold'
    FIELDS_AE = ['barrel', 'sea', 'LNG', 'seabass_hmh', 'noon_jsj']

    # 排除清單儲存目錄 (若無請設為 None 或對應路徑)
    LIST_DIR = 'C:/Users/user/PycharmProjects/meta_expand_test/your_list_folder'

    weight_path = '../output_mix5_SubclassSegFormer_Unified/SubclassSegFormer_Unified_MiT-B0_ripple.pth'

    # --- 2. 載入模型 ---
    model = SubclassSegFormer_Unified(
        backbone='MiT-B0',
        num_classes=2,
        subclass=64,
        num_prompts=5
    ).to(device)

    checkpoint = torch.load(weight_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint.get('state_dict', checkpoint), strict=False)
    model.eval()

    # --- 3. 遍歷 A-E 場域執行推論 ---
    for domain_key in FIELDS_AE:
        print(f"\n======== 開始場域 {domain_key} Baseline 推論 ========")

        # 實例化專為 A-E 設計的測試集
        test_ds = RippleFeatureDataset(
            root=ROOT_AE,
            field=[domain_key],
            is_pure_test=True,
            list_dir=LIST_DIR
        )

        domain_ious = []

        for idx in tqdm(range(len(test_ds)), desc=f"Domain {domain_key}"):
            img_tensor, gt_tensor = test_ds[idx]
            img_tensor = img_tensor.unsqueeze(0).to(device) # (1, 3, 800, 800)
            gt_np = gt_tensor.numpy()                       # (800, 800)

            with torch.no_grad():
                output = model(img_tensor)

                if isinstance(output, tuple):
                    logits = output[0]
                else:
                    logits = output

                # 確保預測與 GT 尺寸一致 (Dataset 已內建 800x800，此處防呆)
                if logits.shape[2:] != gt_tensor.shape:
                    logits = F.interpolate(logits, size=gt_tensor.shape, mode='bilinear', align_corners=False)

                pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

            # --- 4. 計算 IoU ---
            gt_binary = (gt_np > 0).astype(np.uint8)
            intersection = np.logical_and(pred_mask, gt_binary).sum()
            union = np.logical_or(pred_mask, gt_binary).sum()
            iou = intersection / union if union > 0 else 0.0
            domain_ious.append(iou)

        # --- 5. 儲存結果 ---
        os.makedirs('unified_npy', exist_ok=True)
        save_name = f'unified_npy/iou_results_unified_baseline_domain_{domain_key}.base_npy'
        np.save(save_name, np.array(domain_ious))
        print(f"🎉 場域 {domain_key} 完成，平均 IoU: {np.mean(domain_ious):.4f}")

if __name__ == "__main__":
    run_seen_inference_baseline()

# 5 fold Cross validation

In [1]:
import os
import sys

# 1. 指定專案根目錄的絕對路徑
root_path = r"C:\Users\user\PycharmProjects\subclass_segformer"

# 2. 切換工作目錄 (解決讀取 yaml 或圖片時，相對路徑報錯的問題)
os.chdir(root_path)
print("已切換工作目錄至:", os.getcwd())

# 3. 將根目錄加入系統路徑 (解決 import 自定義模組出現 ModuleNotFoundError 的問題)
if root_path not in sys.path:
    sys.path.append(root_path)
    print("已將根目錄加入 sys.path")

已切換工作目錄至: C:\Users\user\PycharmProjects\subclass_segformer


In [3]:
import os
import json
import random
import yaml
from pathlib import Path

# 引入你原本寫好的 dataset.py 中的輔助函數
from dataset import find_source_image

def generate_few_shot_folds():
    # --- 1. 基本設定 ---
    CFG_PATH = 'configs/ripple_prompt.yaml'
    EXPERIMENT_DIR = Path("Subclass/folds_experiment")
    REGISTRY_PATH = EXPERIMENT_DIR / "few_shot_folds_registry.json"

    DOMAINS = ['F', 'G', 'H', 'I', 'J']
    FOLDS_NUM = 5
    SHOTS_PER_FOLD = 5

    EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

    # --- 2. 載入設定檔 ---
    with open(CFG_PATH) as f:
        cfg = yaml.load(f, Loader=yaml.SafeLoader)

    unseen_map = cfg['DATASET'].get('UNSEEN_DOMAINS', {})
    registry = {}

    # 固定 Seed 確保如果誤刪重跑，切出來的 Fold 依然一模一樣
    rng = random.Random(3402)

    # --- 3. 執行資料打亂與切分 ---
    for domain in DOMAINS:
        if domain not in unseen_map:
            print(f"⚠️ 設定檔中找不到場域 {domain} 的路徑，略過。")
            continue

        label_dir = Path(unseen_map[domain])
        all_labels = sorted(list(label_dir.glob('*.png')))

        # 建立配對池 (Image, Label)
        paired_pool = []
        for lbl_path in all_labels:
            img_path = find_source_image(str(lbl_path))
            if img_path:
                paired_pool.append({"img": str(img_path), "lbl": str(lbl_path)})

        total_valid = len(paired_pool)
        required_num = FOLDS_NUM * SHOTS_PER_FOLD

        # 防呆機制：若資料量不足 25 張會給予警告，但程式仍會盡可能切分
        if total_valid < required_num:
            print(f"⚠️ [警告] 場域 {domain} 資料量 ({total_valid} 張) 不足 {required_num} 張，後面的 Fold 可能會少於 {SHOTS_PER_FOLD} 張！")

        # 全局隨機打亂
        rng.shuffle(paired_pool)

        # 切分前 5 個 Fold
        registry[domain] = {}
        for i in range(FOLDS_NUM):
            start_idx = i * SHOTS_PER_FOLD
            end_idx = start_idx + SHOTS_PER_FOLD
            # Python 切片(slice)即使超出索引也不會報錯，會自動取到最後一個元素
            registry[domain][f"fold_{i}"] = paired_pool[start_idx:end_idx]

        print(f"✅ 場域 {domain} 處理完成: 總資料數 {total_valid} 張 | 已提取前 {FOLDS_NUM} 個 Fold。")

    # --- 4. 輸出 JSON 註冊表 ---
    with open(REGISTRY_PATH, 'w', encoding='utf-8') as f:
        json.dump(registry, f, indent=4, ensure_ascii=False)

    print(f"\n📁 資料獨立切分完畢！Fold 註冊表已永久儲存至: {REGISTRY_PATH}")
    return registry

# ==========================================
# 執行切分
# ==========================================
if __name__ == "__main__":
    _ = generate_few_shot_folds()

✅ 場域 F 處理完成: 總資料數 182 張 | 已提取前 5 個 Fold。
✅ 場域 G 處理完成: 總資料數 107 張 | 已提取前 5 個 Fold。
✅ 場域 H 處理完成: 總資料數 131 張 | 已提取前 5 個 Fold。
✅ 場域 I 處理完成: 總資料數 118 張 | 已提取前 5 個 Fold。
✅ 場域 J 處理完成: 總資料數 160 張 | 已提取前 5 個 Fold。

📁 資料獨立切分完畢！Fold 註冊表已永久儲存至: Subclass\folds_experiment\few_shot_folds_registry.json


# 5fold IoU+Precision

In [2]:
import os
import json
import yaml
import glob
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, Subset # 新增 Subset
from tqdm import tqdm
import torch.nn.functional as F

# 載入原有模組
from unified_pipeline import UnifiedInferencePipeline
from dataset import RippleFeatureDataset
from semseg.utils.utils import fix_seeds

def run_5fold_comprehensive_experiment():
    # ==========================================
    # 1. 實驗路徑與參數設定
    # ==========================================
    CFG_PATH = 'configs/ripple_prompt.yaml'
    REGISTRY_PATH = Path("Subclass/folds_experiment/few_shot_folds_registry.json")
    OUTPUT_DIR = Path("Subclass/all_npy")
    CACHE_DIR = Path("Subclass/cache")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    DOMAINS = ['F', 'G', 'H', 'I', 'J']
    MIL_METHODS = ['point', 'box', 'mask']
    FOLDS_NUM = 5

    if not REGISTRY_PATH.exists():
        raise FileNotFoundError(f"❌ 找不到 Fold 註冊表: {REGISTRY_PATH}，請先執行前一個切分腳本！")

    with open(REGISTRY_PATH, 'r', encoding='utf-8') as f:
        registry = json.load(f)

    with open(CFG_PATH) as f:
        cfg = yaml.load(f, Loader=yaml.SafeLoader)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fix_seeds(3402)

    print("⏳ 初始化 unified Pipeline...")
    pipeline = UnifiedInferencePipeline(cfg, device, mil_method='mask')

    # ==========================================
    # 2. 啟動 5-Fold 迴圈
    # ==========================================
    for fold_idx in range(FOLDS_NUM):
        fold_key = f"fold_{fold_idx}"
        print(f"\n{'='*70}")
        print(f"🔥 啟動實驗回合: Round {fold_idx + 1} / {FOLDS_NUM} ({fold_key})")
        print(f"{'='*70}")

        fold_dir = Path(f"Subclass/folds_experiment/{fold_key}")
        fold_dir.mkdir(parents=True, exist_ok=True)

        for domain in DOMAINS:
            txt_path = fold_dir / f"{domain}_mask_hard_samples_val_list.txt"
            with open(txt_path, 'w', encoding='utf-8') as f:
                for sample in registry[domain][fold_key]:
                    basename = os.path.basename(sample['img'])
                    f.write(f"IMG: {basename} | PATH: {sample['img']}\n")

        for domain in DOMAINS:
            print(f"\n>>> 🚀 執行場域: {domain} | Fold: {fold_idx}")

            test_set = RippleFeatureDataset(
                root=cfg['DATASET']['ROOT'], field=[domain], split='test',
                is_pure_test=True, sample_num=5, seed=3402,
                list_dir=str(fold_dir), unseen_map=cfg['DATASET']['UNSEEN_DOMAINS']
            )
            test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

            # --------------------------------------------------
            # 實驗 1: Baseline
            # --------------------------------------------------
            print("   ▶ 正在執行 Baseline...")
            results_base = pipeline.run_baseline_inference(test_loader)
            np.save(OUTPUT_DIR / f"metrics_unified_baseline_fold{fold_idx}_domain_{domain}.base_npy", results_base)

            # --------------------------------------------------
            # 實驗 2: MIL (Point, Box, Mask)
            # --------------------------------------------------
            pipeline.mil_engine.dataset_cfg['LIST_DIR'] = str(fold_dir)

            for mil_m in MIL_METHODS:
                print(f"   ▶ 正在執行 MIL ({mil_m})...")
                cache_file = CACHE_DIR / f"mil_weight_{mil_m}_{domain}.pt"
                if cache_file.exists():
                    cache_file.unlink()

                # ✅ 直接透過參數強制傳入 list_dir=str(fold_dir)
                pipeline.mil_engine.offline_compute_and_save_weights(
                    domain, method=mil_m, need_negative=True, list_dir=str(fold_dir)
                )
                pipeline.mil_engine.load_weights(domain, method=mil_m, need_negative=True)

                pipeline.mil_method = mil_m
                results_mil = pipeline.run_mil_inference(test_loader, field=domain, diagnostic_mode=False)
                np.save(OUTPUT_DIR / f"metrics_unified_mil_{mil_m}_fold{fold_idx}_domain_{domain}.base_npy", results_mil)

            # --------------------------------------------------
            # 實驗 3: Text (Q-prime 直接微調 - 實作 4:1 Split)
            # --------------------------------------------------
            print("   ▶ 正在執行 Text (Q-prime Direct Tuning)...")

            # 讀取完整的 5 張清單
            train_set = RippleFeatureDataset(
                root=cfg['DATASET']['ROOT'], field=[domain], split='train',
                is_pure_test=False, sample_num=5, seed=3402,
                list_dir=str(fold_dir), unseen_map=cfg['DATASET']['UNSEEN_DOMAINS']
            )

            # 使用 Subset 切分 4:1 (前 4 張 Train，最後 1 張 Val)
            dataset_size = len(train_set)
            indices = list(range(dataset_size))
            train_subset = Subset(train_set, indices[:4])
            val_subset = Subset(train_set, indices[4:])

            train_loader = DataLoader(train_subset, batch_size=cfg.get('PROMPT_LEARNING', {}).get('BATCH_SIZE', 2), shuffle=True)
            val_loader = DataLoader(val_subset, batch_size=1, shuffle=False)

            pipeline.model.decode_head.subclass_block.q_prime.data = pipeline.base_q_prime.data.clone().to(device)
            pipeline.model.decode_head.subclass_block.gamma.data = pipeline.base_gamma.data.clone().to(device)

            pipeline.q_tuner.fit(train_loader, val_loader, domain)

            pipeline.model.eval()
            ious_text = []
            precs_text = []
            with torch.no_grad():
                for img, lbl in tqdm(test_loader, desc=f"      [Text] Inference {domain}"):
                    img, lbl = img.to(device), lbl.to(device)
                    output, _ = pipeline.model(img, q_prime=True)

                    if output.shape[2:] != lbl.shape[1:]:
                        output = F.interpolate(output, size=lbl.shape[1:], mode='bilinear', align_corners=False)

                    ious_text.append(pipeline.calculate_iou(output, lbl.squeeze(0)))
                    precs_text.append(pipeline.calculate_precision(output, lbl.squeeze(0)))

            results_text = np.stack((ious_text, precs_text), axis=-1)
            np.save(OUTPUT_DIR / f"metrics_unified_text_fold{fold_idx}_domain_{domain}.base_npy", results_text)

if __name__ == "__main__":
    run_5fold_comprehensive_experiment()
    print("\n🎉 5-Fold 全量實驗已全數執行完畢！結果存放於 Subclass/all_npy。")

⏳ 初始化 Unified Pipeline...
✅ 開始初始化 Pipeline...
✅ 正在載入模型結構與權重...
✅ 模型權重載入完成: output_mix5_SubclassSegFormer_Unified\SubclassSegFormer_Unified_MiT-B0_ripple.pth
✅ 正在初始化 MIL Engine...
✅ 正在初始化 Q-prime Pipeline...
[Debug] 初始化全部完成。

🔥 啟動實驗回合: Round 1 / 5 (fold_0)

>>> 🚀 執行場域: F | Fold: 0
>>> [Standardized Test] F: 讀取清單排除 5 張，剩餘 177 張作為測試集。
   ▶ 正在執行 Baseline...


[Baseline] Inference: 100%|██████████| 177/177 [00:02<00:00, 66.37it/s]


   ▶ 正在執行 MIL (point)...
>>> [Mode: POINT] 正在從場域 F 提取提示並計算權重...
>>> [Fixed Validation] Domain F: 從清單載入 5 張樣本。

🔍 [MIL 提示診斷] 場域 F 實際使用的提示圖片 (共 5 張):
   👉 IMG_6235_cut_269.jpg
   👉 IMG_6235_cut_287.jpg
   👉 IMG_6235_cut_323.jpg
   👉 IMG_6235_cut_4.jpg
   👉 IMG_6235_cut_77.jpg
--------------------------------------------------

✅ [point] 權重存檔: Subclass\cache\mil_weight_point_F.pt | Shape: torch.Size([64])
✅ 載入已固定的少樣本提示權重 (point): Subclass\cache\mil_weight_point_F.pt


[MIL] Inference: 100%|██████████| 177/177 [00:02<00:00, 72.87it/s]


   ▶ 正在執行 MIL (box)...
>>> [Mode: BOX] 正在從場域 F 提取提示並計算權重...
>>> [Fixed Validation] Domain F: 從清單載入 5 張樣本。

🔍 [MIL 提示診斷] 場域 F 實際使用的提示圖片 (共 5 張):
   👉 IMG_6235_cut_269.jpg
   👉 IMG_6235_cut_287.jpg
   👉 IMG_6235_cut_323.jpg
   👉 IMG_6235_cut_4.jpg
   👉 IMG_6235_cut_77.jpg
--------------------------------------------------

✅ [box] 權重存檔: Subclass\cache\mil_weight_box_F.pt | Shape: torch.Size([64])
✅ 載入已固定的少樣本提示權重 (box): Subclass\cache\mil_weight_box_F.pt


...

   ▶ 正在執行 MIL (mask)...
>>> [Mode: MASK] 正在從場域 J 提取提示並計算權重...
>>> [Fixed Validation] Domain J: 從清單載入 5 張樣本。

🔍 [MIL 提示診斷] 場域 J 實際使用的提示圖片 (共 5 張):
   👉 IMG5433_cut_23.jpg
   👉 L_cut_10.jpg
   👉 L_cut_128.jpg
   👉 L_cut_164.jpg
   👉 L_cut_36.jpg
--------------------------------------------------

✅ [mask] 權重存檔: Subclass\cache\mil_weight_mask_J.pt | Shape: torch.Size([64])
✅ 載入已固定的少樣本提示權重 (mask): Subclass\cache\mil_weight_mask_J.pt


[MIL] Inference: 100%|██████████| 155/155 [00:02<00:00, 56.97it/s]


   ▶ 正在執行 Text (Q-prime Direct Tuning)...
>>> [Fixed Training] Domain J: 從清單載入 5 張樣本。
✅ [Tuning] 開始微調 Q-prime 模式
   - 目標場域: J
   - 提示來源: Graph RAG 檢索之已知場域樣本
   Epoch [1/150] | Loss: 0.0136 | RAG-Val IoU: 77.5500 | Gamma: 0.495
   Epoch [50/150] | Loss: 0.0074 | RAG-Val IoU: 80.5000 | Gamma: 0.257
   Epoch [100/150] | Loss: 0.0068 | RAG-Val IoU: 80.6500 | Gamma: 0.140
   Epoch [150/150] | Loss: 0.0062 | RAG-Val IoU: 80.6800 | Gamma: 0.145
✅ 微調完成。最佳 RAG-Val IoU: 80.8000


      [Text] Inference J: 100%|██████████| 155/155 [00:02<00:00, 55.75it/s]


🎉 5-Fold 全量實驗已全數執行完畢！結果存放於 Subclass/all_npy。


# 5fold IoU+Precision MIL alhpa
需要手動去mil_pipeline.py 修改成alpha的計算function
- dealt = MultiInstanceLearning(all_p, all_n)
- optimized_weight = solve_optimal_alpha(dealt, Vb)

In [2]:
import os
import json
import yaml
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.nn.functional as F

# 載入原有模組
from unified_pipeline import UnifiedInferencePipeline
from dataset import RippleFeatureDataset
from semseg.utils.utils import fix_seeds

def run_5fold_mil_only_experiment():
    # ==========================================
    # 1. 實驗路徑與參數設定
    # ==========================================
    CFG_PATH = 'configs/ripple_prompt.yaml'
    REGISTRY_PATH = Path("Subclass/folds_experiment/few_shot_folds_registry.json")
    OUTPUT_DIR = Path("Subclass/all_npy")
    CACHE_DIR = Path("Subclass/cache")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    DOMAINS = ['F', 'G', 'H', 'I', 'J']
    MIL_METHODS = ['point', 'box', 'mask']
    FOLDS_NUM = 5

    if not REGISTRY_PATH.exists():
        raise FileNotFoundError(f"❌ 找不到 Fold 註冊表: {REGISTRY_PATH}，請先執行前一個切分腳本！")

    with open(REGISTRY_PATH, 'r', encoding='utf-8') as f:
        registry = json.load(f)

    with open(CFG_PATH) as f:
        cfg = yaml.load(f, Loader=yaml.SafeLoader)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fix_seeds(3402)

    print("⏳ 初始化 unified Pipeline (僅供 MIL)...")
    # 初始化 Pipeline
    pipeline = UnifiedInferencePipeline(cfg, device, mil_method='mask')

    # ==========================================
    # 2. 啟動 5-Fold 迴圈 (僅執行 MIL)
    # ==========================================
    for fold_idx in range(FOLDS_NUM):
        fold_key = f"fold_{fold_idx}"
        print(f"\n{'='*70}")
        print(f"🔥 啟動實驗回合: Round {fold_idx + 1} / {FOLDS_NUM} ({fold_key})")
        print(f"{'='*70}")

        # --- A. 準備該 Fold 的 TXT 清單 ---
        fold_dir = Path(f"Subclass/folds_experiment/{fold_key}")
        fold_dir.mkdir(parents=True, exist_ok=True)

        for domain in DOMAINS:
            txt_path = fold_dir / f"{domain}_mask_hard_samples_val_list.txt"
            with open(txt_path, 'w', encoding='utf-8') as f:
                for sample in registry[domain][fold_key]:
                    basename = os.path.basename(sample['img'])
                    f.write(f"IMG: {basename} | PATH: {sample['img']}\n")

        # --- B. 遍歷每個場域進行推論 ---
        for domain in DOMAINS:
            print(f"\n>>> 🚀 執行場域: {domain} | Fold: {fold_idx}")

            test_set = RippleFeatureDataset(
                root=cfg['DATASET']['ROOT'], field=[domain], split='test',
                is_pure_test=True, sample_num=5, seed=3402,
                list_dir=str(fold_dir), unseen_map=cfg['DATASET']['UNSEEN_DOMAINS']
            )
            test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

            # --------------------------------------------------
            # 專注於 MIL 實驗 (Point, Box, Mask)
            # --------------------------------------------------
            # 設定 MIL 引擎讀取清單路徑 (雙重保險)
            pipeline.mil_engine.dataset_cfg['LIST_DIR'] = str(fold_dir)

            for mil_m in MIL_METHODS:
                print(f"   ▶ 正在執行 MIL ({mil_m})...")

                # 1. 刪除舊快取，確保針對當下的設定強制重算
                cache_file = CACHE_DIR / f"mil_weight_{mil_m}_{domain}.pt"
                if cache_file.exists():
                    cache_file.unlink()

                # 2. 提取特徵並計算最佳權重 (✅ 已加上 list_dir 強制傳遞)
                pipeline.mil_engine.offline_compute_and_save_weights(
                    domain, method=mil_m, need_negative=True, list_dir=str(fold_dir)
                )
                pipeline.mil_engine.load_weights(domain, method=mil_m, need_negative=True)

                # 3. 執行推論
                pipeline.mil_method = mil_m
                results_mil = pipeline.run_mil_inference(test_loader, field=domain, diagnostic_mode=False)

                # 4. 存檔 (堆疊了 IoU 與 Precision 的陣列)
                np.save(OUTPUT_DIR / f"metrics_unified_mil_alpha_{mil_m}_fold{fold_idx}_domain_{domain}.base_npy", results_mil)

# ==========================================
# 啟動腳本
# ==========================================
if __name__ == "__main__":
    run_5fold_mil_only_experiment()
    print("\n🎉 獨立 MIL 5-Fold 實驗已執行完畢！結果存放於 Subclass/all_npy。")

⏳ 初始化 Unified Pipeline (僅供 MIL)...
✅ 開始初始化 Pipeline...
✅ 正在載入模型結構與權重...
✅ 模型權重載入完成: output_mix5_SubclassSegFormer_Unified\SubclassSegFormer_Unified_MiT-B0_ripple.pth
✅ 正在初始化 MIL Engine...
✅ 正在初始化 Q-prime Pipeline...
[Debug] 初始化全部完成。

🔥 啟動實驗回合: Round 1 / 5 (fold_0)

>>> 🚀 執行場域: F | Fold: 0
>>> [Standardized Test] F: 讀取清單排除 5 張，剩餘 177 張作為測試集。
   ▶ 正在執行 MIL (point)...
>>> [Mode: POINT] 正在從場域 F 提取提示並計算權重...
>>> [Fixed Validation] Domain F: 從清單載入 5 張樣本。

🔍 [MIL 提示診斷] 場域 F 實際使用的提示圖片 (共 5 張):
   👉 IMG_6235_cut_269.jpg
   👉 IMG_6235_cut_287.jpg
   👉 IMG_6235_cut_323.jpg
   👉 IMG_6235_cut_4.jpg
   👉 IMG_6235_cut_77.jpg
--------------------------------------------------

✅ [point] 權重存檔: Subclass\cache\mil_weight_point_F.pt | Shape: torch.Size([64])
✅ 載入已固定的少樣本提示權重 (point): Subclass\cache\mil_weight_point_F.pt


[MIL] Inference: 100%|██████████| 177/177 [00:02<00:00, 63.18it/s]


   ▶ 正在執行 MIL (box)...
>>> [Mode: BOX] 正在從場域 F 提取提示並計算權重...
>>> [Fixed Validation] Domain F: 從清單載入 5 張樣本。

🔍 [MIL 提示診斷] 場域 F 實際使用的提示圖片 (共 5 張):
   👉 IMG_6235_cut_269.jpg
   👉 IMG_6235_cut_287.jpg
   👉 IMG_6235_cut_323.jpg
   👉 IMG_6235_cut_4.jpg
   👉 IMG_6235_cut_77.jpg
--------------------------------------------------

✅ [box] 權重存檔: Subclass\cache\mil_weight_box_F.pt | Shape: torch.Size([64])
✅ 載入已固定的少樣本提示權重 (box): Subclass\cache\mil_weight_box_F.pt


[MIL] Inference: 100%|██████████| 177/177 [00:02<00:00, 64.37it/s]


   ▶ 正在執行 MIL (mask)...
>>> [Mode: MASK] 正在從場域 F 提取提示並計算權重...
>>> [Fixed Validation] Domain F: 從清單載入 5 張樣本。

🔍 [MIL 提示診斷] 場域 F 實際使用的提示圖片 (共 5 張):
   👉 IMG_6235_cut_269.jpg
   👉 IMG_6235_cut_287.jpg
   👉 IMG_6235_cut_323.jpg
   👉 IMG_6235_cut_4.jpg
   👉 IMG_6235_cut_77.jpg
--------------------------------------------------

✅ [mask] 權重存檔: Subclass\cache\mil_weight_mask_F.pt | Shape: torch.Size([64])
✅ 載入已固定的少樣本提示權重 (mask): Subclass\cache\mil_weight_mask_F.pt


...

   ▶ 正在執行 MIL (box)...
>>> [Mode: BOX] 正在從場域 J 提取提示並計算權重...
>>> [Fixed Validation] Domain J: 從清單載入 5 張樣本。

🔍 [MIL 提示診斷] 場域 J 實際使用的提示圖片 (共 5 張):
   👉 IMG5433_cut_23.jpg
   👉 L_cut_10.jpg
   👉 L_cut_128.jpg
   👉 L_cut_164.jpg
   👉 L_cut_36.jpg
--------------------------------------------------

✅ [box] 權重存檔: Subclass\cache\mil_weight_box_J.pt | Shape: torch.Size([64])
✅ 載入已固定的少樣本提示權重 (box): Subclass\cache\mil_weight_box_J.pt


[MIL] Inference: 100%|██████████| 155/155 [00:02<00:00, 59.58it/s]


   ▶ 正在執行 MIL (mask)...
>>> [Mode: MASK] 正在從場域 J 提取提示並計算權重...
>>> [Fixed Validation] Domain J: 從清單載入 5 張樣本。

🔍 [MIL 提示診斷] 場域 J 實際使用的提示圖片 (共 5 張):
   👉 IMG5433_cut_23.jpg
   👉 L_cut_10.jpg
   👉 L_cut_128.jpg
   👉 L_cut_164.jpg
   👉 L_cut_36.jpg
--------------------------------------------------

✅ [mask] 權重存檔: Subclass\cache\mil_weight_mask_J.pt | Shape: torch.Size([64])
✅ 載入已固定的少樣本提示權重 (mask): Subclass\cache\mil_weight_mask_J.pt


[MIL] Inference: 100%|██████████| 155/155 [00:02<00:00, 59.57it/s]


🎉 獨立 MIL 5-Fold 實驗已執行完畢！結果存放於 Subclass/all_npy。


# Case study Auto Evaluation

In [ ]:
import os
import sys

# 1. 指定專案根目錄的絕對路徑
root_path = r"C:\Users\user\PycharmProjects\subclass_segformer"

# 2. 切換工作目錄 (解決讀取 yaml 或圖片時，相對路徑報錯的問題)
os.chdir(root_path)
print("已切換工作目錄至:", os.getcwd())

# 3. 將根目錄加入系統路徑 (解決 import 自定義模組出現 ModuleNotFoundError 的問題)
if root_path not in sys.path:
    sys.path.append(root_path)
    print("已將根目錄加入 sys.path")

In [ ]:
import os
import yaml
import torch
import numpy as np
import random
from pathlib import Path
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from tqdm import tqdm

from unified_pipeline import UnifiedInferencePipeline
from dataset import RippleFeatureDataset


def calculate_scheme_a_score(ious, lambda_penalty=1.5):
    ious = np.array(ious)
    if len(ious) == 0:
        return 0.0, 0.0, 0.0
    mu = np.mean(ious)
    downside_diffs = ious[ious < mu] - mu
    downside_dev = np.mean(np.abs(downside_diffs)) if len(downside_diffs) > 0 else 0.0
    return mu - lambda_penalty * downside_dev, mu, downside_dev


def run_case_study(pipeline_cfg, field, support_list_dir, trials=10, lambda_penalty=1.5):
    """
    固定 5 張提示 (由 support_list_dir 內的清單指定) / 其餘全部測試。
    - test_loader  : is_pure_test=True  → 排除清單內 5 張，剩餘全用
    - support loader: is_pure_test=False → 只讀清單內 5 張
    """
    print(f"\n{'='*60}")
    print(f"🚀 Case Study | 場域: {field}")
    print(f"   提示清單目錄: {support_list_dir}")
    print(f"{'='*60}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pipeline = UnifiedInferencePipeline(pipeline_cfg, device, mil_method='mask')

    save_dir = Path("Subclass/case_study")
    save_dir.mkdir(exist_ok=True)
    log_file = save_dir / f"case_study_domain_{field}.txt"

    # --------------------------------------------------
    # 測試集：排除清單內 5 張，其餘全部當測試
    # --------------------------------------------------
    test_set = RippleFeatureDataset(
        root=pipeline_cfg['DATASET']['ROOT'],
        field=[field],
        split='test',
        is_pure_test=True,
        sample_num=5,
        seed=3402,
        list_dir=support_list_dir,
        unseen_map=pipeline_cfg['DATASET']['UNSEEN_DOMAINS'],
    )
    test_loader = DataLoader(test_set, batch_size=1, shuffle=False)
    query_shots = len(test_set)
    print(f"✅ 測試集就緒: {query_shots} 張")
    assert query_shots > 0, "❌ 測試集是空的，請確認清單路徑與資料夾內容"

    # --------------------------------------------------
    # 提示集：只讀清單內 5 張，供 Q-prime 微調使用
    # --------------------------------------------------
    support_set = RippleFeatureDataset(
        root=pipeline_cfg['DATASET']['ROOT'],
        field=[field],
        split='train',
        is_pure_test=False,
        sample_num=5,
        seed=3402,
        list_dir=support_list_dir,
        unseen_map=pipeline_cfg['DATASET']['UNSEEN_DOMAINS'],
    )
    print(f"✅ 提示集就緒: {len(support_set)} 張")
    assert len(support_set) > 0, "❌ 提示集是空的，請確認清單"

    # MIL 清單路徑已放好，直接用
    mil_methods = ['box', 'point', 'mask']
    cache_dir = Path("Subclass/cache")

    raw_metrics = {
        'Baseline':  np.zeros((trials, query_shots, 2)),
        'MIL_box':   np.zeros((trials, query_shots, 2)),
        'MIL_point': np.zeros((trials, query_shots, 2)),
        'MIL_mask':  np.zeros((trials, query_shots, 2)),
        'Q_prime':   np.zeros((trials, query_shots, 2)),
    }
    stats = {k: {'S': [], 'Mu': [], 'Dev': []} for k in raw_metrics}

    with open(log_file, 'w', encoding='utf-8') as log:
        log.write(f"=== Case Study | 場域: {field} | 提示: 5 張 | 測試: {query_shots} 張 | Trials: {trials} ===\n\n")

        for t in range(trials):
            seed = 3402 + t * 10
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)

            print(f"\n{'-'*50}")
            print(f"🔥 Trial {t+1}/{trials}  (seed={seed})")

            # ── Baseline ──────────────────────────────────
            results_base = pipeline.run_baseline_inference(test_loader)
            raw_metrics['Baseline'][t] = results_base
            s_b, mu_b, dev_b = calculate_scheme_a_score(results_base[:, 0], lambda_penalty)
            stats['Baseline']['S'].append(s_b)
            stats['Baseline']['Mu'].append(mu_b)
            stats['Baseline']['Dev'].append(dev_b)

            # ── MIL (三種提示方式) ─────────────────────────
            pipeline.mil_engine.dataset_cfg['LIST_DIR'] = support_list_dir
            for mil_m in mil_methods:
                cache_file = cache_dir / f"mil_weight_{mil_m}_{field}.pt"
                if cache_file.exists():
                    cache_file.unlink()

                pipeline.mil_engine.offline_compute_and_save_weights(
                    field, method=mil_m, need_negative=True, list_dir=support_list_dir
                )
                pipeline.mil_engine.load_weights(field, method=mil_m, need_negative=True)
                pipeline.mil_method = mil_m

                results_mil = pipeline.run_mil_inference(test_loader, field=field, diagnostic_mode=False)
                key = f'MIL_{mil_m}'
                raw_metrics[key][t] = results_mil
                s_m, mu_m, dev_m = calculate_scheme_a_score(results_mil[:, 0], lambda_penalty)
                stats[key]['S'].append(s_m)
                stats[key]['Mu'].append(mu_m)
                stats[key]['Dev'].append(dev_m)

            # ── Q-prime ───────────────────────────────────
            # 每個 trial 用相同 5 張，只隨機打亂決定 4 train / 1 val
            indices = list(range(len(support_set)))
            random.shuffle(indices)
            train_loader = DataLoader(Subset(support_set, indices[:4]),
                                      batch_size=pipeline_cfg.get('PROMPT_LEARNING', {}).get('BATCH_SIZE', 2),
                                      shuffle=True)
            val_loader   = DataLoader(Subset(support_set, indices[4:]),
                                      batch_size=1, shuffle=False)

            pipeline.model.decode_head.subclass_block.q_prime.data = \
                pipeline.base_q_prime.data.clone().to(device)
            pipeline.model.decode_head.subclass_block.gamma.data = \
                pipeline.base_gamma.data.clone().to(device)
            pipeline.q_tuner.fit(train_loader, val_loader, field)

            pipeline.model.eval()
            q_ious, q_precs = [], []
            with torch.no_grad():
                for img, lbl in tqdm(test_loader, desc=f"   [Q-prime] Trial {t+1}", leave=False):
                    img, lbl = img.to(device), lbl.to(device)
                    output, _ = pipeline.model(img, q_prime=True)
                    if output.shape[2:] != lbl.shape[1:]:
                        output = F.interpolate(output, size=lbl.shape[1:], mode='bilinear', align_corners=False)
                    q_ious.append(pipeline.calculate_iou(output, lbl.squeeze(0)))
                    q_precs.append(pipeline.calculate_precision(output, lbl.squeeze(0)))

            raw_metrics['Q_prime'][t] = np.stack((q_ious, q_precs), axis=-1)
            s_q, mu_q, dev_q = calculate_scheme_a_score(np.array(q_ious), lambda_penalty)
            stats['Q_prime']['S'].append(s_q)
            stats['Q_prime']['Mu'].append(mu_q)
            stats['Q_prime']['Dev'].append(dev_q)

            line = (
                f"[Trial {t+1:02d}] "
                f"Base S={s_b:.4f}(μ={mu_b:.4f}) | "
                f"Box S={stats['MIL_box']['S'][-1]:.4f}(μ={stats['MIL_box']['Mu'][-1]:.4f}) | "
                f"Pt  S={stats['MIL_point']['S'][-1]:.4f}(μ={stats['MIL_point']['Mu'][-1]:.4f}) | "
                f"Msk S={stats['MIL_mask']['S'][-1]:.4f}(μ={stats['MIL_mask']['Mu'][-1]:.4f}) | "
                f"Q'  S={s_q:.4f}(μ={mu_q:.4f})"
            )
            print(f"   {line}")
            log.write(line + "\n")

        # ── 總結 ────────────────────────────────────────
        print(f"\n{'='*50} 總結")
        log.write(f"\n{'='*50} 總結\n")
        for method in stats:
            s   = np.mean(stats[method]['S'])
            mu  = np.mean(stats[method]['Mu'])
            dev = np.mean(stats[method]['Dev'])
            line = f"📈 {method:12s} | S={s:.4f} | IoU={mu:.4f} | 下行風險={dev:.4f}"
            print(line)
            log.write(line + "\n")

        # 儲存 base_npy
        for method, data in raw_metrics.items():
            path = save_dir / f"case_study_{method}_domain_{field}.base_npy"
            np.save(path, data.reshape(-1, 2))
            print(f"💾 {path}")

    print(f"\n✅ 場域 {field} 完畢，日誌: {log_file}")


# ---------------------------------------------------------
if __name__ == "__main__":
    with open('configs/ripple_prompt.yaml') as f:
        cfg = yaml.load(f, Loader=yaml.SafeLoader)
    LIST_DIR = "Subclass/folds_experiment/fold_0"

    for field in ['K', 'L']:
        run_case_study(
            pipeline_cfg=cfg,
            field=field,
            support_list_dir=LIST_DIR,
            trials=10,
            lambda_penalty=1.5,
        )

    print("\n🎉 K、L 兩場域全數完畢！")

In [2]:
"""
text_prompt_dataset.py
========================
獨立模組，新增 TextPromptDataset，專門用於「Text (Graph RAG 模擬) 」實驗。

設計理念：
- 不修改 dataset.py 既有的 RippleFeatureDataset / RippleDataset，
  避免影響已經跑成功的 K/L baseline / MIL / Q-prime 實驗。
- 支援兩種來源結構：
    1. "label_style"  (跟 F~J 一樣): root 指向 unseen_map 的 label 資料夾，
       透過 find_source_image() 反推 image 路徑。
    2. "flat_style"   (B 場域專用): root/<field>/images, root/<field>/labels_detectron2
       平行資料夾，不分 fold，直接用清單檔指定的檔名去比對。

清單檔格式（與 K_mask_hard_samples_val_list.txt 完全相同）：
    IMG: 檔名.jpg | PATH: 完整路徑\檔名.jpg
"""

import os
from pathlib import Path
from torch.utils.data import Dataset
from torchvision import io
import torchvision.transforms.functional as TF
import torch

# 沿用 dataset.py 既有的反推函式，不重複定義
from dataset import find_source_image


class TextPromptDataset(Dataset):
    """
    讀取「文字提示來源場域」（如 B、F）的少量圖片，模擬 Graph RAG 檢索結果。
    僅依賴清單檔，不做任何隨機切分。
    """

    def __init__(self, list_dir: str, source_field: str,
                 image_size=(800, 800),
                 mode: str = "label_style",
                 unseen_map: dict = None,
                 flat_root: str = None):
        """
        Args:
            list_dir: 清單檔所在資料夾
            source_field: 清單檔名前綴，如 'F' 或 'B'
                          (對應 {source_field}_mask_hard_samples_val_list.txt)
            image_size: resize 尺寸，預設與 RippleFeatureDataset 一致 (800, 800)
            mode: "label_style"（跟 F~J 一樣，從 label 反推 image）
                  或 "flat_style"（B 場域：images/ + labels_detectron2/ 平行資料夾，不分 fold）
            unseen_map: 僅 mode="label_style" 時需要，傳入 cfg['DATASET']['UNSEEN_DOMAINS']
            flat_root: 僅 mode="flat_style" 時需要，例如
                       'C:/Users/user/PycharmProjects/organized_ripple_4fold/sea'
                       （該路徑底下需有 images/ 與 labels_detectron2/ 兩個子目錄）
        """
        super().__init__()
        assert mode in ("label_style", "flat_style"), f"未知 mode: {mode}"
        self.list_dir = Path(list_dir)
        self.source_field = source_field
        self.image_size = image_size
        self.mode = mode
        self.unseen_map = unseen_map or {}
        self.flat_root = Path(flat_root) if flat_root else None

        self.files = []  # List[Tuple[img_path, lbl_path]]
        self._load()

        if len(self.files) == 0:
            print(f"⚠️ 警告: TextPromptDataset 場域 {source_field} 讀取結果為 0 張，請檢查清單與路徑設定。")
        else:
            print(f">>> [TextPrompt] 場域 {source_field} ({mode}): 成功載入 {len(self.files)} 張提示影像。")
            for img_p, _ in self.files:
                print(f"    👉 {os.path.basename(img_p)}")

    def _read_list(self):
        """
        讀取清單檔，回傳 List[Tuple[name, full_path]]。
        full_path 是清單裡 PATH 欄位的完整路徑（直接使用，不再二次搜尋）。
        """
        list_file = self.list_dir / f"{self.source_field}_mask_hard_samples_val_list.txt"
        entries = []
        if not list_file.exists():
            raise FileNotFoundError(f"❌ 找不到清單檔: {list_file}")

        with open(list_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if "IMG: " not in line or "| PATH: " not in line:
                    continue
                parts = line.split("| PATH: ")
                name = parts[0].replace("IMG: ", "").strip()
                path = parts[1].strip()
                entries.append((name, path))
        return entries

    def _load(self):
        entries = self._read_list()   # List[Tuple[name, full_path]]

        if self.mode == "label_style":
            # F~J 場域：PATH 指向 label 資料夾內的標註檔，用 find_source_image 反推 image
            if self.source_field not in self.unseen_map:
                raise ValueError(
                    f"❌ mode='label_style' 但 unseen_map 裡找不到場域 {self.source_field}，"
                    f"請確認 cfg['DATASET']['UNSEEN_DOMAINS'] 設定。"
                )
            for name, lbl_path_str in entries:
                lbl_path = Path(lbl_path_str)
                if not lbl_path.exists():
                    print(f"⚠️ 警告: 標註檔不存在 {lbl_path}，略過。")
                    continue
                img_path = find_source_image(str(lbl_path))
                if img_path is None:
                    print(f"⚠️ 警告: 找不到 {lbl_path.name} 對應的原始影像，略過。")
                    continue
                self.files.append((img_path, str(lbl_path)))

        elif self.mode == "flat_style":
            # B 場域：PATH 指向 images/ 底下的圖片，
            # 標註直接換資料夾名稱即可（檔名相同）
            if self.flat_root is None:
                raise ValueError("❌ mode='flat_style' 需要提供 flat_root 參數。")

            labels_dir = self.flat_root / "labels_detectron2"
            if not labels_dir.exists():
                raise FileNotFoundError(f"❌ 找不到標註資料夾: {labels_dir}")

            for name, img_path_str in entries:
                img_path = Path(img_path_str)
                if not img_path.exists():
                    print(f"⚠️ 警告: 影像檔不存在 {img_path}，略過。")
                    continue

                # image 是 .jpg，但標註固定為 .png，只取 stem 換資料夾
                lbl_path = labels_dir / (img_path.stem + ".png")
                if not lbl_path.exists():
                    print(f"⚠️ 警告: 找不到標註 {lbl_path}，略過。")
                    continue

                self.files.append((str(img_path), str(lbl_path)))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        img_path, lbl_path = self.files[index]

        lbl = io.read_image(lbl_path, io.ImageReadMode.GRAY)
        lbl = TF.resize(lbl, list(self.image_size), interpolation=TF.InterpolationMode.NEAREST).squeeze()

        # 標準化標註像素值：
        # B 場域 (sea/labels_detectron2) 標註可能是 0/255，
        # K/L/F 場域通常已是 0/1。
        # 超過 1 的值（排除 ignore_label=255 的語意用途）一律視為前景。
        # 做法：只要 max > 1，就把所有 >0 視為前景 (→1)。
        if lbl.max() > 1:
            lbl = (lbl > 0).long()
        else:
            lbl = lbl.long()

        img = io.read_image(img_path, io.ImageReadMode.RGB)
        img = TF.resize(img, list(self.image_size), interpolation=TF.InterpolationMode.BILINEAR)
        img = TF.convert_image_dtype(img, torch.float)
        img = TF.normalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        return img, lbl

In [3]:
import os
import yaml
import torch
import numpy as np
import random
from pathlib import Path
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from tqdm import tqdm

from unified_pipeline import UnifiedInferencePipeline
from dataset import RippleFeatureDataset


def calculate_scheme_a_score(ious, lambda_penalty=1.5):
    ious = np.array(ious)
    if len(ious) == 0:
        return 0.0, 0.0, 0.0
    mu = np.mean(ious)
    downside_diffs = ious[ious < mu] - mu
    downside_dev = np.mean(np.abs(downside_diffs)) if len(downside_diffs) > 0 else 0.0
    return mu - lambda_penalty * downside_dev, mu, downside_dev


# ==========================================
# K / L 各自的「文字提示來源場域」設定
# ==========================================
TEXT_SOURCE_CONFIG = {
    'K': {
        'source_field': 'F',
        'mode': 'label_style',     # F 跟 K/L 一樣是 label 資料夾 + find_source_image 反推
        'flat_root': None,
    },
    'L': {
        'source_field': 'B',
        'mode': 'flat_style',      # B 是 images/ + labels_detectron2/ 平行資料夾，不分 fold
        'flat_root': 'C:/Users/user/PycharmProjects/organized_ripple_4fold/sea',
    },
}

# B_xxx.txt 與 F_xxx.txt 清單檔所在資料夾（兩個清單都放在 fold_0 底下）
TEXT_LIST_DIR = "Subclass/folds_experiment/fold_0"


def run_case_study(pipeline_cfg, field, support_list_dir, trials=10, lambda_penalty=1.5):
    """
    固定 5 張提示 (由 support_list_dir 內的清單指定) / 其餘全部測試。
    本版本新增第六種方法 Text_RAG：使用「同類型已知場域」(B/F) 的 5 張圖
    模擬 Graph RAG 文字檢索結果，訓練方式與 Q-prime 完全相同，僅資料來源不同。

    - test_loader     : is_pure_test=True  → 排除清單內 5 張，K/L 場域剩餘全用
    - support_set     : is_pure_test=False → K/L 場域清單內 5 張（給 Q-prime 用）
    - text_prompt_set  : 來自 B/F 場域的 5 張（給 Text_RAG 用，與 K/L 場域本身無關）
    """
    print(f"\n{'='*60}")
    print(f"🚀 Case Study | 場域: {field}")
    print(f"   提示清單目錄: {support_list_dir}")
    print(f"{'='*60}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pipeline = UnifiedInferencePipeline(pipeline_cfg, device, mil_method='mask')

    save_dir = Path("Subclass/case_study")
    save_dir.mkdir(exist_ok=True)
    log_file = save_dir / f"case_study_domain_{field}.txt"

    # --------------------------------------------------
    # 測試集：排除清單內 5 張，其餘全部當測試
    # --------------------------------------------------
    test_set = RippleFeatureDataset(
        root=pipeline_cfg['DATASET']['ROOT'],
        field=[field],
        split='test',
        is_pure_test=True,
        sample_num=5,
        seed=3402,
        list_dir=support_list_dir,
        unseen_map=pipeline_cfg['DATASET']['UNSEEN_DOMAINS'],
    )
    test_loader = DataLoader(test_set, batch_size=1, shuffle=False)
    query_shots = len(test_set)
    print(f"✅ 測試集就緒: {query_shots} 張")
    assert query_shots > 0, "❌ 測試集是空的，請確認清單路徑與資料夾內容"

    # --------------------------------------------------
    # 提示集：只讀清單內 5 張，供 Q-prime 微調使用
    # --------------------------------------------------
    support_set = RippleFeatureDataset(
        root=pipeline_cfg['DATASET']['ROOT'],
        field=[field],
        split='train',
        is_pure_test=False,
        sample_num=5,
        seed=3402,
        list_dir=support_list_dir,
        unseen_map=pipeline_cfg['DATASET']['UNSEEN_DOMAINS'],
    )
    print(f"✅ 提示集就緒: {len(support_set)} 張")
    assert len(support_set) > 0, "❌ 提示集是空的，請確認清單"

    # --------------------------------------------------
    # 文字提示集 (Text_RAG 模擬)：來自 B/F 等同類型已知場域的 5 張
    # --------------------------------------------------
    text_cfg = TEXT_SOURCE_CONFIG[field]
    text_prompt_set = TextPromptDataset(
        list_dir=TEXT_LIST_DIR,
        source_field=text_cfg['source_field'],
        image_size=tuple(pipeline_cfg['TRAIN']['IMAGE_SIZE']),
        mode=text_cfg['mode'],
        unseen_map=pipeline_cfg['DATASET']['UNSEEN_DOMAINS'],
        flat_root=text_cfg['flat_root'],
    )
    print(f"✅ 文字提示集就緒 (來源場域 {text_cfg['source_field']}): {len(text_prompt_set)} 張")
    assert len(text_prompt_set) > 0, (
        f"❌ 文字提示集是空的！場域 {field} 對應來源 {text_cfg['source_field']}，"
        f"請確認清單檔 {TEXT_LIST_DIR}/{text_cfg['source_field']}_mask_hard_samples_val_list.txt 是否存在。"
    )

    # MIL 清單路徑已放好，直接用
    mil_methods = ['box', 'point', 'mask']
    cache_dir = Path("Subclass/cache")

    raw_metrics = {
        'Baseline':  np.zeros((trials, query_shots, 2)),
        'MIL_box':   np.zeros((trials, query_shots, 2)),
        'MIL_point': np.zeros((trials, query_shots, 2)),
        'MIL_mask':  np.zeros((trials, query_shots, 2)),
        'Q_prime':   np.zeros((trials, query_shots, 2)),
        'Text_RAG':  np.zeros((trials, query_shots, 2)),
    }
    stats = {k: {'S': [], 'Mu': [], 'Dev': []} for k in raw_metrics}

    with open(log_file, 'w', encoding='utf-8') as log:
        log.write(
            f"=== Case Study | 場域: {field} | 提示: 5 張 | 測試: {query_shots} 張 | Trials: {trials} "
            f"| Text 提示來源: {text_cfg['source_field']} ===\n\n"
        )

        for t in range(trials):
            seed = 3402 + t * 10
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)

            print(f"\n{'-'*50}")
            print(f"🔥 Trial {t+1}/{trials}  (seed={seed})")

            # ── Baseline ──────────────────────────────────
            results_base = pipeline.run_baseline_inference(test_loader)
            raw_metrics['Baseline'][t] = results_base
            s_b, mu_b, dev_b = calculate_scheme_a_score(results_base[:, 0], lambda_penalty)
            stats['Baseline']['S'].append(s_b)
            stats['Baseline']['Mu'].append(mu_b)
            stats['Baseline']['Dev'].append(dev_b)

            # ── MIL (三種提示方式) ─────────────────────────
            pipeline.mil_engine.dataset_cfg['LIST_DIR'] = support_list_dir
            for mil_m in mil_methods:
                cache_file = cache_dir / f"mil_weight_{mil_m}_{field}.pt"
                if cache_file.exists():
                    cache_file.unlink()

                pipeline.mil_engine.offline_compute_and_save_weights(
                    field, method=mil_m, need_negative=True, list_dir=support_list_dir
                )
                pipeline.mil_engine.load_weights(field, method=mil_m, need_negative=True)
                pipeline.mil_method = mil_m

                results_mil = pipeline.run_mil_inference(test_loader, field=field, diagnostic_mode=False)
                key = f'MIL_{mil_m}'
                raw_metrics[key][t] = results_mil
                s_m, mu_m, dev_m = calculate_scheme_a_score(results_mil[:, 0], lambda_penalty)
                stats[key]['S'].append(s_m)
                stats[key]['Mu'].append(mu_m)
                stats[key]['Dev'].append(dev_m)

            # ── Q-prime（場域自己的 5 張）───────────────────
            indices = list(range(len(support_set)))
            random.shuffle(indices)
            train_loader = DataLoader(Subset(support_set, indices[:4]),
                                      batch_size=pipeline_cfg.get('PROMPT_LEARNING', {}).get('BATCH_SIZE', 2),
                                      shuffle=True)
            val_loader   = DataLoader(Subset(support_set, indices[4:]),
                                      batch_size=1, shuffle=False)

            pipeline.model.decode_head.subclass_block.q_prime.data = \
                pipeline.base_q_prime.data.clone().to(device)
            pipeline.model.decode_head.subclass_block.gamma.data = \
                pipeline.base_gamma.data.clone().to(device)
            pipeline.q_tuner.fit(train_loader, val_loader, field)

            pipeline.model.eval()
            q_ious, q_precs = [], []
            with torch.no_grad():
                for img, lbl in tqdm(test_loader, desc=f"   [Q-prime] Trial {t+1}", leave=False):
                    img, lbl = img.to(device), lbl.to(device)
                    output, _ = pipeline.model(img, q_prime=True)
                    if output.shape[2:] != lbl.shape[1:]:
                        output = F.interpolate(output, size=lbl.shape[1:], mode='bilinear', align_corners=False)
                    q_ious.append(pipeline.calculate_iou(output, lbl.squeeze(0)))
                    q_precs.append(pipeline.calculate_precision(output, lbl.squeeze(0)))

            raw_metrics['Q_prime'][t] = np.stack((q_ious, q_precs), axis=-1)
            s_q, mu_q, dev_q = calculate_scheme_a_score(np.array(q_ious), lambda_penalty)
            stats['Q_prime']['S'].append(s_q)
            stats['Q_prime']['Mu'].append(mu_q)
            stats['Q_prime']['Dev'].append(dev_q)

            # ── Text_RAG（B/F 場域的 5 張，模擬 Graph RAG 檢索結果）──
            # 訓練邏輯與 Q-prime 完全相同：4:1 train/val split + 同一個 fit()
            print(f"   ▶ 正在執行 Text_RAG (來源場域: {text_cfg['source_field']})...")

            text_indices = list(range(len(text_prompt_set)))
            random.shuffle(text_indices)
            text_train_loader = DataLoader(
                Subset(text_prompt_set, text_indices[:4]),
                batch_size=pipeline_cfg.get('PROMPT_LEARNING', {}).get('BATCH_SIZE', 2),
                shuffle=True,
            )
            text_val_loader = DataLoader(
                Subset(text_prompt_set, text_indices[4:]),
                batch_size=1, shuffle=False,
            )

            # 重置 q_prime / gamma 到初始值（與 Q-prime 實驗起點一致，公平比較）
            pipeline.model.decode_head.subclass_block.q_prime.data = \
                pipeline.base_q_prime.data.clone().to(device)
            pipeline.model.decode_head.subclass_block.gamma.data = \
                pipeline.base_gamma.data.clone().to(device)
            pipeline.q_tuner.fit(text_train_loader, text_val_loader, field)

            pipeline.model.eval()
            text_ious, text_precs = [], []
            with torch.no_grad():
                for img, lbl in tqdm(test_loader, desc=f"   [Text_RAG] Trial {t+1}", leave=False):
                    img, lbl = img.to(device), lbl.to(device)
                    output, _ = pipeline.model(img, q_prime=True)
                    if output.shape[2:] != lbl.shape[1:]:
                        output = F.interpolate(output, size=lbl.shape[1:], mode='bilinear', align_corners=False)
                    text_ious.append(pipeline.calculate_iou(output, lbl.squeeze(0)))
                    text_precs.append(pipeline.calculate_precision(output, lbl.squeeze(0)))

            raw_metrics['Text_RAG'][t] = np.stack((text_ious, text_precs), axis=-1)
            s_t, mu_t, dev_t = calculate_scheme_a_score(np.array(text_ious), lambda_penalty)
            stats['Text_RAG']['S'].append(s_t)
            stats['Text_RAG']['Mu'].append(mu_t)
            stats['Text_RAG']['Dev'].append(dev_t)

            line = (
                f"[Trial {t+1:02d}] "
                f"Base S={s_b:.4f}(μ={mu_b:.4f}) | "
                f"Box S={stats['MIL_box']['S'][-1]:.4f}(μ={stats['MIL_box']['Mu'][-1]:.4f}) | "
                f"Pt  S={stats['MIL_point']['S'][-1]:.4f}(μ={stats['MIL_point']['Mu'][-1]:.4f}) | "
                f"Msk S={stats['MIL_mask']['S'][-1]:.4f}(μ={stats['MIL_mask']['Mu'][-1]:.4f}) | "
                f"Q'  S={s_q:.4f}(μ={mu_q:.4f}) | "
                f"Text S={s_t:.4f}(μ={mu_t:.4f})"
            )
            print(f"   {line}")
            log.write(line + "\n")

        # ── 總結 ────────────────────────────────────────
        print(f"\n{'='*50} 總結")
        log.write(f"\n{'='*50} 總結\n")
        for method in stats:
            s   = np.mean(stats[method]['S'])
            mu  = np.mean(stats[method]['Mu'])
            dev = np.mean(stats[method]['Dev'])
            line = f"📈 {method:12s} | S={s:.4f} | IoU={mu:.4f} | 下行風險={dev:.4f}"
            print(line)
            log.write(line + "\n")

        # 儲存 base_npy
        for method, data in raw_metrics.items():
            path = save_dir / f"case_study_{method}_domain_{field}.base_npy"
            np.save(path, data.reshape(-1, 2))
            print(f"💾 {path}")

    print(f"\n✅ 場域 {field} 完畢，日誌: {log_file}")


# ---------------------------------------------------------
if __name__ == "__main__":
    with open('configs/ripple_prompt.yaml') as f:
        cfg = yaml.load(f, Loader=yaml.SafeLoader)
    LIST_DIR = "Subclass/folds_experiment/fold_0"

    for field in ['K', 'L']:
        run_case_study(
            pipeline_cfg=cfg,
            field=field,
            support_list_dir=LIST_DIR,
            trials=10,
            lambda_penalty=1.5,
        )

    print("\n🎉 K、L 兩場域全數完畢！")


🚀 Case Study | 場域: K
   提示清單目錄: Subclass/folds_experiment/fold_0
✅ 開始初始化 Pipeline...
✅ 正在載入模型結構與權重...
✅ 模型權重載入完成: output_mix5_SubclassSegFormer_Unified\SubclassSegFormer_Unified_MiT-B0_ripple.pth
✅ 正在初始化 MIL Engine...
✅ 正在初始化 Q-prime Pipeline...
[Debug] 初始化全部完成。
>>> [Standardized Test] K: 讀取清單排除 5 張，剩餘 10 張作為測試集。
✅ 測試集就緒: 10 張
>>> [Fixed Training] Domain K: 從清單載入 5 張樣本。
✅ 提示集就緒: 5 張
>>> [TextPrompt] 場域 F (label_style): 成功載入 5 張提示影像。
    👉 IMG_6235_cut_4.jpg
    👉 IMG_6235_cut_269.jpg
    👉 IMG_6235_cut_287.jpg
    👉 IMG_6235_cut_77.jpg
    👉 IMG_6235_cut_323.jpg
✅ 文字提示集就緒 (來源場域 F): 5 張

--------------------------------------------------
🔥 Trial 1/10  (seed=3402)


[Baseline] Inference: 100%|██████████| 10/10 [00:00<00:00, 19.98it/s]


>>> [Mode: BOX] 正在從場域 K 提取提示並計算權重...
>>> [Fixed Validation] Domain K: 從清單載入 5 張樣本。

🔍 [MIL 提示診斷] 場域 K 實際使用的提示圖片 (共 5 張):
   👉 IMG_6749_002088.jpg
   👉 IMG_6749_002089.jpg
   👉 IMG_6749_002090.jpg
   👉 IMG_6749_002091.jpg
   👉 IMG_6749_002092.jpg
--------------------------------------------------

✅ [box] 權重存檔: Subclass\cache\mil_weight_box_K.pt | Shape: torch.Size([64])
✅ 載入已固定的少樣本提示權重 (box): Subclass\cache\mil_weight_box_K.pt


[MIL] Inference: 100%|██████████| 10/10 [00:00<00:00, 39.19it/s]


>>> [Mode: POINT] 正在從場域 K 提取提示並計算權重...
>>> [Fixed Validation] Domain K: 從清單載入 5 張樣本。

🔍 [MIL 提示診斷] 場域 K 實際使用的提示圖片 (共 5 張):
   👉 IMG_6749_002088.jpg
   👉 IMG_6749_002089.jpg
   👉 IMG_6749_002090.jpg
   👉 IMG_6749_002091.jpg
   👉 IMG_6749_002092.jpg
--------------------------------------------------

✅ [point] 權重存檔: Subclass\cache\mil_weight_point_K.pt | Shape: torch.Size([64])
✅ 載入已固定的少樣本提示權重 (point): Subclass\cache\mil_weight_point_K.pt


...

>>> [Mode: MASK] 正在從場域 L 提取提示並計算權重...
>>> [Fixed Validation] Domain L: 從清單載入 5 張樣本。

🔍 [MIL 提示診斷] 場域 L 實際使用的提示圖片 (共 5 張):
   👉 IMG_6836_002406.jpg
   👉 IMG_6836_002407.jpg
   👉 IMG_6836_002408.jpg
   👉 IMG_6836_002409.jpg
   👉 IMG_6836_002410.jpg
--------------------------------------------------

✅ [mask] 權重存檔: Subclass\cache\mil_weight_mask_L.pt | Shape: torch.Size([64])
✅ 載入已固定的少樣本提示權重 (mask): Subclass\cache\mil_weight_mask_L.pt


[MIL] Inference: 100%|██████████| 10/10 [00:00<00:00, 42.55it/s]


✅ [Tuning] 開始微調 Q-prime 模式
   - 目標場域: L
   - 提示來源: Graph RAG 檢索之已知場域樣本
   Epoch [1/150] | Loss: 0.0380 | RAG-Val IoU: 60.2800 | Gamma: 0.495
   Epoch [50/150] | Loss: 0.0175 | RAG-Val IoU: 59.8900 | Gamma: 0.252
   Epoch [100/150] | Loss: 0.0089 | RAG-Val IoU: 62.7700 | Gamma: 0.078
   Epoch [150/150] | Loss: 0.0090 | RAG-Val IoU: 62.6700 | Gamma: 0.078
✅ 微調完成。最佳 RAG-Val IoU: 62.9700


   ▶ 正在執行 Text_RAG (來源場域: B)...
✅ [Tuning] 開始微調 Q-prime 模式
   - 目標場域: L
   - 提示來源: Graph RAG 檢索之已知場域樣本
   Epoch [1/150] | Loss: 0.0059 | RAG-Val IoU: 93.7200 | Gamma: 0.495
   Epoch [50/150] | Loss: 0.0063 | RAG-Val IoU: 92.6000 | Gamma: 0.336
   Epoch [100/150] | Loss: 0.0048 | RAG-Val IoU: 92.7900 | Gamma: 0.337
   Epoch [150/150] | Loss: 0.0045 | RAG-Val IoU: 92.7100 | Gamma: 0.330
✅ 微調完成。最佳 RAG-Val IoU: 93.7500


   [Trial 10] Base S=0.5697(μ=0.6519) | Box S=0.3992(μ=0.4569) | Pt  S=0.5487(μ=0.6232) | Msk S=0.5039(μ=0.5741) | Q'  S=0.5740(μ=0.6833) | Text S=0.5779(μ=0.6623)

================================================== 總結
📈 Baseline     | S=0.5673 | IoU=0.6475 | 下行風險=0.0535
📈 MIL_box      | S=0.3980 | IoU=0.4666 | 下行風險=0.0457
📈 MIL_point    | S=0.5536 | IoU=0.6319 | 下行風險=0.0522
📈 MIL_mask     | S=0.5046 | IoU=0.5817 | 下行風險=0.0515
📈 Q_prime      | S=0.5765 | IoU=0.6824 | 下行風險=0.0706
📈 Text_RAG     | S=0.5775 | IoU=0.6608 | 下行風險=0.0556
💾 Subclass\eval_npy\case_study_Baseline_domain_L.npy
💾 Subclass\eval_npy\case_study_MIL_box_domain_L.npy
💾 Subclass\eval_npy\case_study_MIL_point_domain_L.npy
💾 Subclass\eval_npy\case_study_MIL_mask_domain_L.npy
💾 Subclass\eval_npy\case_study_Q_prime_domain_L.npy
💾 Subclass\eval_npy\case_study_Text_RAG_domain_L.npy

✅ 場域 L 完畢，日誌: Subclass\eval_npy\case_study_domain_L.txt

🎉 K、L 兩場域全數完畢！
